# CAN-SHIELD reproduction

Reproduces the experiments in "A Novelty Shortcut in CAN Intrusion Detection: Fuzzing Attacks in Training Destroy Cross-Vehicle Transfer, and Protocol-Level Features Are Robust to It" (Hassan et al.).

Put the dataset files in `PROJECT_DIR/data/`:

```
DoS_dataset.csv
Fuzzy_dataset.csv
gear_dataset.csv
RPM_dataset.csv
Attack_free_dataset.txt
DoS_attack_dataset.txt
Fuzzy_attack_dataset.txt
Impersonation_attack_dataset.txt   (optional)
```

Runs on a Colab CPU runtime. Finished steps are checkpointed under `results_v2/checkpoints/`, so after a disconnect just run all cells again.

In [ ]:
PROJECT_DIR = "/content/drive/MyDrive/Can Shield/reproduce"

MASTER_SEED = 42
MULTI_SEEDS = [0, 1, 2, 3, 4]
E3_SEEDS = MULTI_SEEDS
E5_SEEDS = MULTI_SEEDS

WINDOW = 64
PAD_TO = 8

E1_POOL = 214288
E1_CV_POOL = 50000
E2_TOTAL = 120000
E3_BASE_EACH = 40000
E3_FUZZ_SUB = 40000
E3_DOSES = [0, 2, 5, 10, 25, 50, 100]
E5_TOTAL = 150002
RF_TRANSFER_TREES = 60
RF_E1_TREES = 120

OTIDS_TEST_TOTAL = 2573117

SHAP_N_FRAMES = 250
SHAP_N_BACKGROUND = 200
SHAP_PAIRED = True
SHAP_N_PERM = SHAP_N_BACKGROUND if SHAP_PAIRED else 40
SHAP_BACKGROUND = "real_balanced"
SHAP_TREESHAP_CHECK = True
LOCAL_PICK_RULE = "medoid"
PERM_IMP_REPEATS = 10
PERM_IMP_SAMPLE = 20000

FORCE_RECOMPUTE = []
RUN_SLOW_E1 = True
EMIT_SUPPLEMENTARY = True
RUN_DEDUP_SENSITIVITY = True
VERIFY_SHA256 = False
KNN_ALGORITHM = "brute"

import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    if not os.path.ismount("/content/drive"):
        drive.mount("/content/drive")

try:
    import imblearn
except ImportError:
    print("Installing imbalanced-learn ...")
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "imbalanced-learn"], check=False)

PROJECT_DIR = PROJECT_DIR.rstrip("/\\")

if not os.path.isdir(PROJECT_DIR):
    print("PROJECT_DIR does not exist:\n   " + PROJECT_DIR + "\n")
    _parent = os.path.dirname(PROJECT_DIR)
    while _parent and not os.path.isdir(_parent):
        _parent = os.path.dirname(_parent)
    if _parent and os.path.isdir(_parent):
        print("What is actually inside " + _parent + ":")
        for _e in sorted(os.listdir(_parent))[:60]:
            _tag = "[dir] " if os.path.isdir(os.path.join(_parent, _e)) else "      "
            print("   " + _tag + _e)
    raise FileNotFoundError(
        "PROJECT_DIR not found: " + PROJECT_DIR + "\n"
        "Copy the exact folder name from the listing above into PROJECT_DIR. "
        "Capitalisation and spaces matter, and the Drive root is 'MyDrive' (no space)."
    )

DATA_DIR = os.path.join(PROJECT_DIR, "data")
RESULTS = os.path.join(PROJECT_DIR, "results_v2")
TABLE_DIR = os.path.join(RESULTS, "tables")
FIG_DIR = os.path.join(RESULTS, "figures")
SUPP_DIR = os.path.join(RESULTS, "supplementary")
CACHE_DIR = os.path.join(RESULTS, "cache")
CKPT_DIR = os.path.join(RESULTS, "checkpoints")
LOG_DIR = os.path.join(RESULTS, "logs")
for _d in (RESULTS, TABLE_DIR, FIG_DIR, CACHE_DIR, CKPT_DIR, LOG_DIR):
    os.makedirs(_d, exist_ok=True)
if EMIT_SUPPLEMENTARY:
    os.makedirs(SUPP_DIR, exist_ok=True)

print("Project :", PROJECT_DIR)
print("Data    :", DATA_DIR, " exists:", os.path.isdir(DATA_DIR))
print("Results :", RESULTS)

In [ ]:
import json, re, time, math, hashlib, logging, warnings, itertools, gc, platform, shutil
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch
import sklearn
from sklearn.ensemble import (RandomForestClassifier, AdaBoostClassifier,
                              GradientBoostingClassifier, StackingClassifier)
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.inspection import permutation_importance
import scipy
from scipy import stats
try:
    from sklearn.ensemble import HistGradientBoostingClassifier
except ImportError:
    from sklearn.experimental import enable_hist_gradient_boosting
    from sklearn.ensemble import HistGradientBoostingClassifier
import imblearn
from imblearn.over_sampling import SMOTE

warnings.filterwarnings("ignore")
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 80)

logger = logging.getLogger("canshield")
logger.setLevel(logging.INFO)
logger.handlers = []
_fh = logging.FileHandler(os.path.join(LOG_DIR, "run.log"), encoding="utf-8")
_fh.setFormatter(logging.Formatter("%(asctime)s  %(message)s"))
logger.addHandler(_fh)

def log(*a):
    msg = " ".join(str(x) for x in a)
    print(msg, flush=True)
    logger.info(msg)

ENV = {
    "utc_start": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "scikit_learn": sklearn.__version__,
    "imblearn": imblearn.__version__,
    "scipy": scipy.__version__,
    "cpu_count": os.cpu_count(),
}
log("ENVIRONMENT")
for _k, _v in ENV.items():
    log(f"    {_k:14s} {_v}")

C_ENG, C_RAW, C_HI, C_NORM, C_ATK = "#1b7f79", "#8a8f94", "#e07b39", "#2f6f9f", "#b5443a"
plt.rcParams.update({
    "figure.dpi": 110, "savefig.dpi": 300, "savefig.bbox": "tight",
    "font.size": 10, "axes.titlesize": 11, "axes.labelsize": 10,
    "axes.grid": True, "grid.alpha": 0.25, "grid.linewidth": 0.6,
    "axes.spines.top": False, "axes.spines.right": False, "legend.frameon": False,
})

def save_fig(fig, name, caption=""):
    for ext in ("png", "pdf"):
        fig.savefig(os.path.join(FIG_DIR, f"{name}.{ext}"))
    log(f"    [figure] {name}.png/.pdf   {caption}")
    plt.show()
    plt.close(fig)

def _write_table(df, directory, name, caption, float_fmt, index, tag):
    df.to_csv(os.path.join(directory, f"{name}.csv"), index=index)
    try:
        with open(os.path.join(directory, f"{name}.tex"), "w", encoding="utf-8") as f:
            f.write(df.to_latex(index=index,
                                float_format=lambda v: float_fmt % v,
                                caption=caption or name, label="tab:" + name))
    except Exception as e:
        log("    (LaTeX export skipped:", e, ")")
    log(f"    [{tag}] {name}.csv/.tex   rows={len(df)}   {caption}")
    return df

def save_table(df, name, caption="", float_fmt="%.4f", index=False):
    return _write_table(df, TABLE_DIR, name, caption, float_fmt, index, "table")

def save_supp(df, name, caption="", float_fmt="%.4f", index=False):
    if not EMIT_SUPPLEMENTARY:
        return df
    return _write_table(df, SUPP_DIR, name, caption, float_fmt, index, "supp ")

In [ ]:
class NpEncoder(json.JSONEncoder):
    def default(self, o):
        if isinstance(o, (np.integer,)): return int(o)
        if isinstance(o, (np.floating,)): return float(o)
        if isinstance(o, (np.bool_,)): return bool(o)
        if isinstance(o, np.ndarray): return o.tolist()
        return super().default(o)

def _atomic_write_bytes(path, writer):
    tmp = path + ".tmp"
    with open(tmp, "wb") as f:
        writer(f)
        f.flush()
        os.fsync(f.fileno())
    os.replace(tmp, path)

class Ckpt:
    def __init__(self, root, force_prefixes=()):
        self.root = root
        self.force = tuple(force_prefixes)
        self.manifest_path = os.path.join(root, "manifest.json")
        self.manifest = {}
        if os.path.exists(self.manifest_path):
            try:
                with open(self.manifest_path, encoding="utf-8") as f:
                    self.manifest = json.load(f)
            except Exception:
                self.manifest = {}

    def _safe(self, key):
        return key.replace("/", "__").replace(" ", "_")

    def jpath(self, key): return os.path.join(self.root, self._safe(key) + ".json")
    def npath(self, key): return os.path.join(self.root, self._safe(key) + ".npz")

    def _forced(self, key):
        return any(key.startswith(p) for p in self.force)

    def _touch(self, key, kind, secs):
        self.manifest[key] = {"kind": kind, "seconds": round(secs, 2),
                              "utc": datetime.now(timezone.utc).isoformat(timespec="seconds")}
        _atomic_write_bytes(self.manifest_path,
                            lambda f: f.write(json.dumps(self.manifest, indent=1,
                                                         cls=NpEncoder).encode("utf-8")))

    def run(self, key, fn, quiet=False):
        p = self.jpath(key)
        if os.path.exists(p) and not self._forced(key):
            with open(p, encoding="utf-8") as f:
                if not quiet:
                    print(f"    [skip] {key}")
                return json.load(f)
        t0 = time.time()
        if not quiet:
            print(f"    [run ] {key} ...", end="", flush=True)
        out = fn()
        _atomic_write_bytes(p, lambda f: f.write(json.dumps(out, indent=1,
                                                            cls=NpEncoder).encode("utf-8")))
        dt = time.time() - t0
        self._touch(key, "json", dt)
        if not quiet:
            print(f" done in {dt:,.1f}s")
        logger.info(f"[unit] {key} {dt:.1f}s")
        return out

    def run_np(self, key, fn):
        p = self.npath(key)
        if os.path.exists(p) and not self._forced(key):
            print(f"    [skip] {key}")
            return dict(np.load(p, allow_pickle=False))
        t0 = time.time()
        print(f"    [run ] {key} ...", end="", flush=True)
        out = fn()
        _atomic_write_bytes(p, lambda f: np.savez(f, **out))
        dt = time.time() - t0
        self._touch(key, "npz", dt)
        print(f" done in {dt:,.1f}s")
        logger.info(f"[unit] {key} {dt:.1f}s")
        return out

    def done(self, key):
        return (os.path.exists(self.jpath(key)) or os.path.exists(self.npath(key))) \
               and not self._forced(key)

ckpt = Ckpt(CKPT_DIR, FORCE_RECOMPUTE)
log(f"Checkpoint store: {CKPT_DIR}")
log(f"   units already complete: {len(ckpt.manifest)}")
if FORCE_RECOMPUTE:
    log(f"   FORCE_RECOMPUTE prefixes: {FORCE_RECOMPUTE}")

def cache_npy(name, build_fn, dtype=None):
    p = os.path.join(CACHE_DIR, name + ".npy")
    if not os.path.exists(p):
        t0 = time.time()
        print(f"    [build] cache/{name}.npy ...", end="", flush=True)
        arr = build_fn()
        if dtype is not None:
            arr = arr.astype(dtype, copy=False)
        tmp = p + ".tmp.npy"
        np.save(tmp, arr)
        os.replace(tmp, p)
        print(f" {arr.shape} {arr.dtype}  {time.time()-t0:,.1f}s")
        del arr
        gc.collect()
    return np.load(p, mmap_mode="r")

def cache_exists(name):
    return os.path.exists(os.path.join(CACHE_DIR, name + ".npy"))

## Parsing and feature extraction

In [ ]:
CAR_HACKING = {
    "DoS": "DoS_dataset.csv",
    "Fuzzy": "Fuzzy_dataset.csv",
    "gear": "gear_dataset.csv",
    "RPM": "RPM_dataset.csv",
}
OTIDS = {
    "attack_free": "Attack_free_dataset.txt",
    "dos": "DoS_attack_dataset.txt",
    "fuzzy": "Fuzzy_attack_dataset.txt",
    "impersonation": "Impersonation_attack_dataset.txt",
}
CH_ORDER = ["DoS", "Fuzzy", "gear", "RPM"]
CH_PRETTY = {"DoS": "DoS (flooding)", "Fuzzy": "Fuzzy", "gear": "Gear spoof", "RPM": "RPM spoof"}
OPTIONAL = {"impersonation"}
DATA_EXTS = (".csv", ".txt")

def _norm(name):
    stem, ext = os.path.splitext(name)
    for ch in (" ", "_", "-", "."):
        stem = stem.replace(ch, "")
    return stem.lower() + ext.lower()

def _scan_project_files():
    found = []
    skip = {os.path.normpath(RESULTS)}
    for root, dirs, files in os.walk(PROJECT_DIR):
        dirs[:] = [d for d in dirs
                   if os.path.normpath(os.path.join(root, d)) not in skip
                   and not d.startswith(".")]
        for f in files:
            if f.lower().endswith(DATA_EXTS) and not f.startswith("."):
                found.append(os.path.join(root, f))
    return sorted(found)

SCANNED = _scan_project_files()
_by_exact = {}
_by_norm = {}
for _p in SCANNED:
    _b = os.path.basename(_p)
    _by_exact.setdefault(_b.lower(), []).append(_p)
    _by_norm.setdefault(_norm(_b), []).append(_p)

def _prefer(cands):
    if not cands:
        return None
    def rank(p):
        in_data = os.path.normpath(os.path.dirname(p)) == os.path.normpath(DATA_DIR)
        depth = os.path.normpath(p).count(os.sep)
        return (0 if in_data else 1, depth, -os.path.getsize(p))
    return sorted(cands, key=rank)[0]

def _resolve(expected):
    hit = _prefer(_by_exact.get(expected.lower(), []))
    if hit:
        return hit, ("exact" if os.path.basename(hit) == expected else "case-insensitive")
    hit = _prefer(_by_norm.get(_norm(expected), []))
    if hit:
        return hit, "normalised"
    return None, "not found"

RESOLVED_BY_NAME, _rows = {}, []
for _group, _mapping in (("Car-Hacking", CAR_HACKING), ("OTIDS", OTIDS)):
    for _key, _fn in _mapping.items():
        _path, _how = _resolve(_fn)
        if _path:
            RESOLVED_BY_NAME[_fn] = _path
        _rows.append({
            "group": _group, "capture": _key, "expected_file": _fn,
            "present": _path is not None,
            "matched_by": _how,
            "found_as": os.path.relpath(_path, PROJECT_DIR) if _path else "",
            "MB": round(os.path.getsize(_path) / 1e6, 1) if _path else 0.0,
            "optional": _key in OPTIONAL,
        })

inv_df = pd.DataFrame(_rows)
display(inv_df[["group", "capture", "expected_file", "present",
                "matched_by", "found_as", "MB", "optional"]])

def dpath(fn):
    p = RESOLVED_BY_NAME.get(fn)
    if p is None:
        raise FileNotFoundError(f"{fn} was not resolved - re-run this cell")
    return p

def sha256_of(path, chunk=1 << 22):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(chunk), b""):
            h.update(b)
    return h.hexdigest()

missing = [r for r in _rows if not r["present"] and not r["optional"]]
if missing:
    import difflib
    names_seen = sorted({os.path.basename(p) for p in SCANNED})
    print("\n" + "=" * 78)
    print("REQUIRED DATASET FILES NOT FOUND")
    print("=" * 78)
    print("PROJECT_DIR : " + PROJECT_DIR)
    print("data/       : " + DATA_DIR +
          ("   (exists)" if os.path.isdir(DATA_DIR) else "   (DOES NOT EXIST)"))
    print("\nSearched PROJECT_DIR recursively and found "
          + str(len(SCANNED)) + " .csv/.txt file(s):")
    if SCANNED:
        for p in SCANNED[:60]:
            print(f"   {os.path.relpath(p, PROJECT_DIR):<58s} {os.path.getsize(p)/1e6:8.1f} MB")
        if len(SCANNED) > 60:
            print(f"   ... and {len(SCANNED)-60} more")
    else:
        print("   (none at all - the files are elsewhere, or still uploading)")
    print("\nTop level of PROJECT_DIR:")
    try:
        for e in sorted(os.listdir(PROJECT_DIR))[:60]:
            tag = "[dir] " if os.path.isdir(os.path.join(PROJECT_DIR, e)) else "      "
            print("   " + tag + e)
    except OSError as e:
        print("   cannot list: " + str(e))
    print("\nStill missing:")
    for r in missing:
        near = difflib.get_close_matches(r["expected_file"], names_seen, n=3, cutoff=0.5)
        print(f"   {r['expected_file']}")
        if near:
            print(f"      closest names present: {', '.join(near)}")
    print("=" * 78)
    raise FileNotFoundError(
        "Missing: " + ", ".join(r["expected_file"] for r in missing) + "\n"
        "Check the listing above. Usual causes: the files are in a different folder than "
        "PROJECT_DIR; they are still uploading (a part-uploaded file shows a smaller size); "
        "or a name differs by more than case and punctuation, in which case rename it to the "
        "expected name. Then re-run THIS cell - nothing here is cached."
    )

HAVE_IMPERSONATION = "Impersonation_attack_dataset.txt" in RESOLVED_BY_NAME
log("All required inputs resolved.")
for r in _rows:
    if r["present"]:
        log(f"   {r['capture']:14s} {r['found_as']:<46s} {r['MB']:8.1f} MB  ({r['matched_by']})")
log(f"   impersonation capture available: {HAVE_IMPERSONATION}"
    + ("" if HAVE_IMPERSONATION else "  (optional; the Section 6.1 impersonation check is skipped)"))

if VERIFY_SHA256:
    for r in _rows:
        if r["present"]:
            k = "stage1/sha256/" + r["capture"]
            d = ckpt.run(k, lambda r=r: {"file": r["found_as"],
                                         "bytes": os.path.getsize(dpath(r["expected_file"])),
                                         "sha256": sha256_of(dpath(r["expected_file"]))},
                         quiet=True)
            log(f"   sha256 {r['capture']:14s} {d['sha256']}")

In [ ]:
_HEX_LUT = np.full(256, -1, dtype=np.int16)
_HEX_LUT[np.frombuffer(b"0123456789", np.uint8)] = np.arange(10)
_HEX_LUT[np.frombuffer(b"abcdef", np.uint8)] = np.arange(10, 16)
_HEX_LUT[np.frombuffer(b"ABCDEF", np.uint8)] = np.arange(10, 16)

def hex_to_int(a, width):
    a = np.asarray(a, dtype=np.str_)
    shape = a.shape
    s = np.char.zfill(a.ravel(), width).astype("S" + str(width))
    b = np.frombuffer(s.tobytes(), dtype=np.uint8).reshape(-1, width)
    d = _HEX_LUT[b]
    if (d < 0).any():
        bad = s[(d < 0).any(axis=1)][:5]
        raise ValueError(f"non-hexadecimal token(s) encountered: {bad}")
    out = np.zeros(d.shape[0], dtype=np.int64)
    for k in range(width):
        out = out * 16 + d[:, k]
    return out.reshape(shape)

def parse_car_hacking(path, chunksize=500_000):
    ts_l, cid_l, dlc_l, b8_l, y_l = [], [], [], [], []
    reader = pd.read_csv(path, header=None, names=list(range(12)), dtype=str,
                         na_filter=False, chunksize=chunksize, engine="c")
    for chunk in reader:
        dlc = chunk[2].to_numpy(dtype=np.int64)
        arr = chunk.iloc[:, 3:12].to_numpy(dtype="U2")
        n = len(dlc)
        label = arr[np.arange(n), dlc]
        tok = arr[:, :PAD_TO].copy()
        tok[np.arange(PAD_TO)[None, :] >= dlc[:, None]] = "00"
        ts_l.append(chunk[0].to_numpy(dtype=np.float64))
        cid_l.append(hex_to_int(chunk[1].to_numpy(dtype="U4"), 4).astype(np.int16))
        dlc_l.append(dlc.astype(np.int8))
        b8_l.append(hex_to_int(tok, 2).astype(np.uint8))
        y_l.append((label == "T").astype(np.int8))
        if not np.isin(label, ["R", "T"]).all():
            raise ValueError(f"unexpected label value in {path}: "
                             f"{np.unique(label[~np.isin(label, ['R','T'])])[:5]}")
    return (np.concatenate(ts_l), np.concatenate(cid_l), np.concatenate(dlc_l),
            np.concatenate(b8_l), np.concatenate(y_l))

OTIDS_BASE_COLS = 24
OTIDS_MAX_COLS = 256

def _read_whitespace(path, ncols, chunksize):
    kw = dict(header=None, names=list(range(ncols)), dtype=str,
              na_filter=False, chunksize=chunksize, engine="c")
    try:
        return pd.read_csv(path, sep=r"\s+", **kw)
    except (ValueError, TypeError):
        return pd.read_csv(path, delim_whitespace=True, **kw)

def _parse_otids_pass(path, ncols, label_rule, chunksize):
    ts_l, cid_l, dlc_l, b8_l = [], [], [], []
    rem_l, tot_l, bad_l = [], [], []
    for chunk in _read_whitespace(path, ncols, chunksize):
        tot_l.append(len(chunk))
        dlc = pd.to_numeric(chunk[6], errors="coerce").to_numpy()
        valid = ~np.isnan(dlc)
        dlc = np.where(valid, dlc, -1).astype(np.int64)
        rem_l.append(int((valid & (dlc == 0)).sum()))
        bad_l.append(int((~valid | (dlc > PAD_TO)).sum()))
        keep = valid & (dlc > 0) & (dlc <= PAD_TO)
        if not keep.any():
            continue
        sub = chunk[keep]
        dlc = dlc[keep]
        tok = sub.iloc[:, 7:7 + PAD_TO].to_numpy(dtype="U2")
        tok[np.arange(PAD_TO)[None, :] >= dlc[:, None]] = "00"
        ts_l.append(sub[1].to_numpy(dtype=np.float64))
        cid_l.append(hex_to_int(sub[3].to_numpy(dtype="U4"), 4).astype(np.int16))
        dlc_l.append(dlc.astype(np.int8))
        b8_l.append(hex_to_int(tok, 2).astype(np.uint8))
    if not ts_l:
        raise ValueError(f"no parsable data frames found in {path}")
    ts = np.concatenate(ts_l)
    cid = np.concatenate(cid_l)
    dlc = np.concatenate(dlc_l)
    b8 = np.concatenate(b8_l)
    y = (label_rule(cid, ts) if label_rule is not None
           else np.zeros(len(ts), dtype=np.int8))
    meta = {"lines_total": int(sum(tot_l)), "remote_frames_dropped": int(sum(rem_l)),
            "malformed_lines_dropped": int(sum(bad_l)),
            "data_frames": int(len(ts)), "columns_allocated": int(ncols)}
    return ts, cid, dlc, b8, y.astype(np.int8), meta

def parse_otids(path, label_rule=None, chunksize=500_000, ncols=OTIDS_BASE_COLS):
    while True:
        try:
            return _parse_otids_pass(path, ncols, label_rule, chunksize)
        except pd.errors.ParserError as e:
            m = re.search(r"saw\s+(\d+)", str(e))
            need = (int(m.group(1)) + 8) if m else ncols * 2
            if need <= ncols or need > OTIDS_MAX_COLS:
                raise
            log(f"    (a line in {os.path.basename(path)} carries more tokens than expected; "
                f"widening the parse to {need} columns and re-reading)")
            ncols = need

def otids_dos_rule(cid, ts):
    return (cid == 0).astype(np.int8)

log("Parsers ready (Stage S decoding, Stage H harmonisation).")

In [ ]:
FEATURES = [
    ("CAN_ID", "Raw", "S", "Arbitration identifier decoded to decimal"),
    ("DLC", "Raw", "S", "Data length code (0-8)"),
    ("DATA0", "Raw", "H", "Payload byte 0, zero-padded above DLC-1"),
    ("DATA1", "Raw", "H", "Payload byte 1, zero-padded above DLC-1"),
    ("DATA2", "Raw", "H", "Payload byte 2, zero-padded above DLC-1"),
    ("DATA3", "Raw", "H", "Payload byte 3, zero-padded above DLC-1"),
    ("DATA4", "Raw", "H", "Payload byte 4, zero-padded above DLC-1"),
    ("DATA5", "Raw", "H", "Payload byte 5, zero-padded above DLC-1"),
    ("DATA6", "Raw", "H", "Payload byte 6, zero-padded above DLC-1"),
    ("DATA7", "Raw", "H", "Payload byte 7, zero-padded above DLC-1"),
    ("IAT", "Engineered", "I", "Global inter-arrival time (s)"),
    ("IAT_ID", "Engineered", "I", "Inter-arrival time since the previous frame of the same identifier (s)"),
    ("ID_Freq", "Engineered", "L", "Identifier repetitions in a 64-frame sliding window"),
    ("Payload_Entropy", "Engineered", "E", "Shannon entropy of the payload byte-value distribution (0-3 bits)"),
    ("Payload_Hamming", "Engineered", "E", "Byte-wise Hamming distance to the previous payload of the same identifier"),
    ("NonZero_Bytes", "Engineered", "E", "Count of non-zero payload bytes"),
    ("Data_Mean", "Engineered", "E", "Mean of the eight payload byte values"),
    ("Data_Std", "Engineered", "E", "Standard deviation of the eight payload byte values"),
]
FEATURE_NAMES = [f[0] for f in FEATURES]
RAW_10 = [i for i, f in enumerate(FEATURES) if f[1] == "Raw"]
ENG_8 = [i for i, f in enumerate(FEATURES) if f[1] == "Engineered"]
ALL_18 = list(range(len(FEATURES)))
REPRESENTATIONS = {"raw10": RAW_10, "all18": ALL_18, "eng8": ENG_8}
REP_PRETTY = {"raw10": "Raw 10", "all18": "CAN-SHIELD 18", "eng8": "Engineered 8"}
assert len(RAW_10) == 10 and len(ENG_8) == 8 and len(FEATURES) == 18

def payload_entropy(b8, chunk=1_000_000):
    n = len(b8)
    out = np.empty(n, dtype=np.float32)
    for s in range(0, n, chunk):
        b = np.asarray(b8[s:s + chunk])
        m = len(b)
        sb = np.sort(b, axis=1)
        new = np.ones((m, PAD_TO), dtype=bool)
        new[:, 1:] = sb[:, 1:] != sb[:, :-1]
        runid = np.cumsum(new, axis=1) - 1
        flat = (np.arange(m, dtype=np.int64)[:, None] * PAD_TO + runid).ravel()
        cnt = np.bincount(flat, minlength=PAD_TO * m).reshape(m, PAD_TO)
        p = cnt / float(PAD_TO)
        with np.errstate(divide="ignore", invalid="ignore"):
            term = np.where(cnt > 0, p * np.log2(p), 0.0)
        out[s:s + chunk] = -term.sum(axis=1)
    return out

def engineer(ts, cid, dlc, b8, window=WINDOW):
    n = len(ts)
    iat = np.empty(n, dtype=np.float64)
    iat[0] = 0.0
    iat[1:] = np.diff(ts)

    iat_id = np.zeros(n, dtype=np.float64)
    ham = np.zeros(n, dtype=np.int16)
    id_freq = np.zeros(n, dtype=np.int32)

    order = np.argsort(cid, kind="stable")
    cs = cid[order]
    starts = np.flatnonzero(np.concatenate(([True], cs[1:] != cs[:-1])))
    bounds = np.concatenate((starts, [n]))

    for a, b in zip(bounds[:-1], bounds[1:]):
        idx = order[a:b]
        t = ts[idx]
        d = np.empty(len(t), dtype=np.float64)
        d[0] = 0.0
        if len(t) > 1:
            d[1:] = np.diff(t)
        iat_id[idx] = d
        p = np.asarray(b8[idx])
        h = np.zeros(len(t), dtype=np.int16)
        if len(t) > 1:
            h[1:] = (p[1:] != p[:-1]).sum(axis=1)
        ham[idx] = h
        pos = idx.astype(np.int64)
        lo = np.searchsorted(pos, pos - window + 1, side="left")
        id_freq[idx] = (np.arange(len(pos)) - lo + 1).astype(np.int32)

    b8a = np.asarray(b8)
    ent = payload_entropy(b8a)
    nz = (b8a != 0).sum(axis=1).astype(np.float32)
    dmean = b8a.mean(axis=1, dtype=np.float32)
    dstd = b8a.std(axis=1, dtype=np.float32)

    X = np.empty((n, 18), dtype=np.float32)
    X[:, 0] = cid
    X[:, 1] = dlc
    X[:, 2:10] = b8a
    X[:, 10] = iat
    X[:, 11] = iat_id
    X[:, 12] = id_freq
    X[:, 13] = ent
    X[:, 14] = ham
    X[:, 15] = nz
    X[:, 16] = dmean
    X[:, 17] = dstd
    return X

log("Feature pipeline ready.")
log("   raw 10        :", [FEATURE_NAMES[i] for i in RAW_10])
log("   engineered 8  :", [FEATURE_NAMES[i] for i in ENG_8])

In [ ]:
CAPTURE_STATS = {}
CH_TS, OT_TS = {}, {}

def _build_capture(key, fn, kind):
    name_X, name_y, name_t = f"X_{kind}_{key}", f"y_{kind}_{key}", f"ts_{kind}_{key}"
    if not (cache_exists(name_X) and cache_exists(name_y) and cache_exists(name_t)):
        log(f"  parsing {fn} ...")
        t0 = time.time()
        if kind == "ch":
            ts, cid, dlc, b8, y = parse_car_hacking(dpath(fn))
            meta = {"lines_total": int(len(ts)), "remote_frames_dropped": 0,
                    "data_frames": int(len(ts))}
        else:
            rule = otids_dos_rule if key == "dos" else None
            ts, cid, dlc, b8, y, meta = parse_otids(dpath(fn), label_rule=rule)
        log(f"    parsed {len(ts):,} data frames in {time.time()-t0:,.1f}s "
            f"(remote dropped: {meta['remote_frames_dropped']:,})")
        t0 = time.time()
        X = engineer(ts, cid, dlc, b8)
        log(f"    engineered 18 features in {time.time()-t0:,.1f}s")
        cache_npy(name_X, lambda: X)
        cache_npy(name_y, lambda: y)
        cache_npy(name_t, lambda: ts, dtype=np.float64)
        ckpt.run(f"stage1/parse_meta/{kind}_{key}", lambda: meta, quiet=True)
        del ts, cid, dlc, b8, y, X
        gc.collect()
    X = cache_npy(name_X, lambda: None)
    y = cache_npy(name_y, lambda: None)
    t = cache_npy(name_t, lambda: None)
    (CH_TS if kind == "ch" else OT_TS)[key] = t
    return X, y

log("=== Car-Hacking (Hyundai YF Sonata) ===")
CH = {}
for key in CH_ORDER:
    CH[key] = _build_capture(key, CAR_HACKING[key], "ch")
    X, y = CH[key]
    log(f"  {key:6s} X={X.shape} y attack={int(y.sum()):,} / {len(y):,} "
        f"({100*y.mean():.2f}%)")

log("=== OTIDS (KIA Soul) ===")
OT = {}
for key in ["attack_free", "dos", "fuzzy"]:
    OT[key] = _build_capture(key, OTIDS[key], "ot")
    X, y = OT[key]
    log(f"  {key:12s} X={X.shape} y attack={int(y.sum()):,} / {len(y):,} "
        f"({100*y.mean():.2f}%)")
if HAVE_IMPERSONATION:
    OT["impersonation"] = _build_capture("impersonation", OTIDS["impersonation"], "ot")

log("Feature cache complete.  Cached files:")
for f in sorted(os.listdir(CACHE_DIR)):
    log(f"   {f:28s} {os.path.getsize(os.path.join(CACHE_DIR,f))/1e6:8.1f} MB")

## Input checks

In [ ]:
def _id_vocab():
    out = {}
    for key in CH_ORDER:
        X, y = CH[key]
        cid = np.asarray(X[:, 0]).astype(np.int32)
        ya = np.asarray(y).astype(bool)
        out[f"ch_{key}"] = {
            "frames": int(len(cid)),
            "attack_frames": int(ya.sum()),
            "attack_pct": float(100 * ya.mean()),
            "attack_ids": sorted(np.unique(cid[ya]).tolist()),
            "normal_ids": sorted(np.unique(cid[~ya]).tolist()),
            "all_ids_n": int(len(np.unique(cid))),
        }
    an, nn = set(), set()
    fr = at = 0
    for key in CH_ORDER:
        d = out[f"ch_{key}"]
        an |= set(d["attack_ids"]); nn |= set(d["normal_ids"])
        fr += d["frames"]; at += d["attack_frames"]
    out["ch_combined"] = {"frames": fr, "attack_frames": at,
                          "attack_pct": float(100 * at / fr),
                          "attack_ids": sorted(an), "normal_ids": sorted(nn),
                          "all_ids_n": len(an | nn)}
    for key in ["attack_free", "dos", "fuzzy"]:
        X, y = OT[key]
        cid = np.asarray(X[:, 0]).astype(np.int32)
        ya = np.asarray(y).astype(bool)
        out[f"ot_{key}"] = {
            "frames": int(len(cid)),
            "attack_frames": int(ya.sum()),
            "attack_pct": float(100 * ya.mean()),
            "attack_ids": sorted(np.unique(cid[ya]).tolist()),
            "normal_ids": sorted(np.unique(cid[~ya]).tolist()),
            "all_ids_n": int(len(np.unique(cid))),
        }
    return out

VOCAB = ckpt.run("stage2/id_vocabulary", _id_vocab)

DUP_SCAN_LIMIT = 1_500_000

def _dup_audit():
    keys = CH_ORDER
    cols = list(range(10))

    def tail(k, m):
        X, y = CH[k]
        n = len(y)
        raw = np.asarray(X[n - m:, :][:, cols], dtype=np.float64)
        ts = np.asarray(CH_TS[k][n - m:], dtype=np.float64).reshape(-1, 1)
        return np.hstack([ts, raw])[::-1]

    pairs = {}
    for a, b in itertools.combinations(keys, 2):
        m = min(len(CH[a][1]), len(CH[b][1]), DUP_SCAN_LIMIT)
        eq = (tail(a, m) == tail(b, m)).all(axis=1)
        c = int(m if eq.all() else np.argmin(eq))
        pairs[f"{a}|{b}"] = {"common_suffix_rows": c, "hit_scan_limit": bool(c == m)}
    shared = min(v["common_suffix_rows"] for v in pairs.values())
    X, y = CH["RPM"]
    n = len(y)
    suf_y = np.asarray(y[n - shared:]) if shared else np.array([], dtype=np.int8)
    total = sum(VOCAB[f"ch_{k}"]["frames"] for k in keys)
    dup = shared * (len(keys) - 1)
    return {
        "pairwise": pairs,
        "shared_suffix_rows": int(shared),
        "shared_suffix_attack_frames": int(suf_y.sum()),
        "shared_suffix_normal_frames": int(len(suf_y) - suf_y.sum()),
        "captures_sharing_it": len(keys),
        "combined_frames": int(total),
        "duplicate_frames": int(dup),
        "duplicate_pct_of_corpus": float(100 * dup / total),
        "unique_frames": int(total - dup),
        "suffix_fraction_of_capture": {k: float(shared / VOCAB[f"ch_{k}"]["frames"])
                                       for k in keys},
    }

DUP = ckpt.run("stage2/duplication_audit", _dup_audit)

log("--- cross-capture duplication ---")
for k, v in DUP["pairwise"].items():
    log(f"   {k:20s} identical common suffix = {v['common_suffix_rows']:>9,} rows")
log(f"   shared suffix                 : {DUP['shared_suffix_rows']:,} frames, "
    f"present in all {DUP['captures_sharing_it']} captures")
log(f"   its label composition         : {DUP['shared_suffix_normal_frames']:,} normal / "
    f"{DUP['shared_suffix_attack_frames']:,} attack")
log(f"   exact duplicate frames        : {DUP['duplicate_frames']:,} "
    f"({DUP['duplicate_pct_of_corpus']:.2f}% of the combined corpus)")
log(f"   unique frames                 : {DUP['unique_frames']:,} "
    f"(paper reports {DUP['combined_frames']:,} labelled frames)")
log("   suffix as a fraction of each capture:")
for k, v in DUP["suffix_fraction_of_capture"].items():
    log(f"      {k:6s} {100*v:5.2f}%  -> "
        f"{'lands inside the last 30% (chronological test side)' if v < 0.30 else 'crosses the 70/30 boundary'}")

dup_rows = [{"pair": k, "identical_common_suffix_rows": v["common_suffix_rows"]}
            for k, v in DUP["pairwise"].items()]
dup_rows.append({"pair": "SHARED BY ALL FOUR", "identical_common_suffix_rows": DUP["shared_suffix_rows"]})
save_supp(pd.DataFrame(dup_rows), "tableA1_duplication_audit",
           "Identical common suffix shared by the Car-Hacking captures", float_fmt="%.0f")

save_supp(pd.DataFrame([{
    "combined_frames_as_published": DUP["combined_frames"],
    "exact_duplicate_frames":       DUP["duplicate_frames"],
    "duplicate_pct_of_corpus":      round(DUP["duplicate_pct_of_corpus"], 2),
    "unique_frames":                DUP["unique_frames"],
    "shared_suffix_rows":           DUP["shared_suffix_rows"],
    "shared_suffix_all_normal":     DUP["shared_suffix_attack_frames"] == 0,
}]), "tableA2_corpus_size_audit", "Effective corpus size after de-duplication", float_fmt="%.2f")

In [ ]:
def _build_otids_testset():
    Xd, yd = OT["dos"]
    Xa, ya = OT["attack_free"]
    nd, na = len(yd), len(ya)
    need = OTIDS_TEST_TOTAL - nd
    if need < 0:
        raise ValueError(f"OTIDS DoS capture alone ({nd:,}) exceeds the target "
                         f"{OTIDS_TEST_TOTAL:,}")
    if need > na:
        log(f"   NOTE: only {na:,} attack-free data frames available, {need:,} requested; "
            f"using all of them.")
        need = na
    rs = np.random.RandomState(MASTER_SEED)
    sel = np.sort(rs.choice(na, size=need, replace=False))
    return {"attack_free_index": sel.astype(np.int64),
            "n_dos": np.array([nd]), "n_attack_free": np.array([need])}

_ots = ckpt.run_np("stage2/otids_testset_index", _build_otids_testset)
_af_idx = _ots["attack_free_index"]

def load_otids_test():
    Xd, yd = OT["dos"]
    Xa, ya = OT["attack_free"]
    X = np.concatenate([np.asarray(Xd), np.asarray(Xa)[_af_idx]], axis=0)
    y = np.concatenate([np.asarray(yd), np.asarray(ya)[_af_idx]], axis=0)
    return X, y

X_OT, y_OT = load_otids_test()
OTIDS_TEST = {
    "frames": int(len(y_OT)),
    "attack_frames": int(y_OT.sum()),
    "attack_pct": float(100 * y_OT.mean()),
    "from_dos_capture": int(_ots["n_dos"][0]),
    "from_attack_free": int(_ots["n_attack_free"][0]),
}
ckpt.run("stage2/otids_testset_stats", lambda: OTIDS_TEST, quiet=True)
log("Cross-vehicle test set (KIA Soul):")
for k, v in OTIDS_TEST.items():
    log(f"   {k:18s} {v:,}" if isinstance(v, int) else f"   {k:18s} {v:.4f}")

def _overlap():
    ot_norm = set(VOCAB["ot_attack_free"]["normal_ids"])
    rows = []
    def add(label, s, note):
        rows.append({"car_hacking_scope": label,
                     "ch_normal_ids": len(s),
                     "otids_normal_ids": len(ot_norm),
                     "shared": len(s & ot_norm),
                     "ch_only_unfamiliar_to_otids_model": len(s - ot_norm),
                     "otids_only": len(ot_norm - s),
                     "note": note})
    add("combined (all four captures)", set(VOCAB["ch_combined"]["normal_ids"]),
        "quoted in Section 3.2")
    for k in CH_ORDER:
        add(f"{k} capture alone", set(VOCAB[f"ch_{k}"]["normal_ids"]),
            "E4 test set" if k == "DoS" else "")
    return rows

OVERLAP = ckpt.run("stage2/id_overlap", _overlap)
ov_df = pd.DataFrame(OVERLAP)
display(ov_df)
save_supp(ov_df, "tableA3_identifier_overlap",
           "Normal-traffic identifier vocabulary overlap, Car-Hacking vs OTIDS", float_fmt="%.0f")

In [ ]:
def _table2():
    rows = []
    for k in CH_ORDER:
        d = VOCAB[f"ch_{k}"]
        rows.append({"dataset_vehicle": "Car-Hacking, Hyundai YF Sonata",
                     "capture": CH_PRETTY[k],
                     "frames": d["frames"], "attack_frames": d["attack_frames"],
                     "attack_pct": round(d["attack_pct"], 2),
                     "distinct_attack_class_ids": len(d["attack_ids"]),
                     "distinct_normal_class_ids": len(d["normal_ids"]),
                     "distinct_ids_in_capture": d["all_ids_n"]})
    d = VOCAB["ch_combined"]
    rows.append({"dataset_vehicle": "Car-Hacking, Hyundai YF Sonata", "capture": "Combined",
                 "frames": d["frames"], "attack_frames": d["attack_frames"],
                 "attack_pct": round(d["attack_pct"], 2),
                 "distinct_attack_class_ids": len(d["attack_ids"]),
                 "distinct_normal_class_ids": len(d["normal_ids"]),
                 "distinct_ids_in_capture": d["all_ids_n"]})
    rows.append({"dataset_vehicle": "OTIDS, KIA Soul", "capture": "DoS + attack-free",
                 "frames": OTIDS_TEST["frames"], "attack_frames": OTIDS_TEST["attack_frames"],
                 "attack_pct": round(OTIDS_TEST["attack_pct"], 2),
                 "distinct_attack_class_ids": len(VOCAB["ot_dos"]["attack_ids"]),
                 "distinct_normal_class_ids": len(VOCAB["ot_attack_free"]["normal_ids"]),
                 "distinct_ids_in_capture": VOCAB["ot_attack_free"]["all_ids_n"]})
    return rows

t2 = pd.DataFrame(ckpt.run("tables/table2", _table2))
save_table(t2, "table02_dataset_composition",
           "Datasets after parsing and remote-frame removal", float_fmt="%.2f")
display(t2)

In [ ]:
def allocate(total, weights, keys):
    w = np.asarray(weights, dtype=np.float64)
    w = w / w.sum()
    exact = total * w
    base = np.floor(exact).astype(np.int64)
    rem = total - base.sum()
    if rem:
        base[np.argsort(-(exact - base))[:rem]] += 1
    return dict(zip(keys, base.tolist()))

CAPTURE_WEIGHTS = [VOCAB[f"ch_{k}"]["frames"] for k in CH_ORDER]

DEDUP_KEEPER = "RPM"
DEDUP_TAIL = {k: (0 if k == DEDUP_KEEPER else int(DUP["shared_suffix_rows"])) for k in CH_ORDER}

def stratified_take(key, m, seed, exclude_tail=0):
    X, y = CH[key]
    y = np.asarray(y)
    valid = len(y) - int(exclude_tail)
    ai = np.flatnonzero(y[:valid] == 1)
    ni = np.flatnonzero(y[:valid] == 0)
    n = valid
    mt = int(round(m * len(ai) / n))
    mt = min(mt, len(ai))
    mn = min(m - mt, len(ni))
    rs = np.random.RandomState(seed)
    sel = np.concatenate([rs.choice(ai, mt, replace=False),
                          rs.choice(ni, mn, replace=False)])
    sel.sort()
    return np.asarray(X)[sel], y[sel]

def build_training_set(spec, seed, dedup=False):
    Xs, ys = [], []
    for i, (k, m) in enumerate(sorted(spec.items())):
        if m <= 0:
            continue
        Xi, yi = stratified_take(k, m, seed * 1000 + 7 * i + 13,
                                 exclude_tail=DEDUP_TAIL[k] if dedup else 0)
        Xs.append(Xi); ys.append(yi)
    X = np.concatenate(Xs, axis=0)
    y = np.concatenate(ys, axis=0)
    rs = np.random.RandomState(seed)
    p = rs.permutation(len(y))
    return X[p], y[p]

def train_attack_id_count(X, y):
    ya = np.asarray(y).astype(bool)
    return int(len(np.unique(np.asarray(X)[ya, 0].astype(np.int32))))

BALANCE_POLICY = {"raw10": "none", "eng8": "smote", "all18": "smote"}
SCALE_POLICY = {"raw10": False, "eng8": True, "all18": True}

def prepare(Xtr, ytr, rep, balance=None, seed=MASTER_SEED):
    cols = REPRESENTATIONS[rep]
    Xtr = np.asarray(Xtr)[:, cols].astype(np.float32)
    ytr = np.asarray(ytr)
    scaler = None
    if SCALE_POLICY[rep]:
        scaler = StandardScaler().fit(Xtr)
        Xtr = scaler.transform(Xtr).astype(np.float32)
    mode = BALANCE_POLICY[rep] if balance is None else balance
    cw = None
    if mode == "smote" and len(np.unique(ytr)) > 1:
        k = max(1, min(5, int(np.bincount(ytr).min()) - 1))
        Xtr, ytr = SMOTE(random_state=seed, k_neighbors=k).fit_resample(Xtr, ytr)
    elif mode == "class_weight":
        cw = "balanced"
    return Xtr, ytr, scaler, cw, cols

def apply_prepare(Xte, cols, scaler):
    Xte = np.asarray(Xte)[:, cols].astype(np.float32)
    return scaler.transform(Xte).astype(np.float32) if scaler is not None else Xte

def make_model(name, seed=MASTER_SEED, class_weight=None, n_trees=None):
    cw = class_weight
    if name == "LogisticRegression":
        return LogisticRegression(penalty="l2", C=1.0, max_iter=2000,
                                  random_state=seed, class_weight=cw, n_jobs=None)
    if name == "KNeighbours":
        return KNeighborsClassifier(n_neighbors=5, algorithm=KNN_ALGORITHM, n_jobs=-1)
    if name == "DecisionTree":
        return DecisionTreeClassifier(max_depth=12, random_state=seed, class_weight=cw)
    if name == "RandomForest":
        return RandomForestClassifier(n_estimators=n_trees or RF_E1_TREES, random_state=seed,
                                      class_weight=cw, n_jobs=-1)
    if name == "AdaBoost":
        return AdaBoostClassifier(random_state=seed)
    if name == "GradientBoosting":
        return GradientBoostingClassifier(random_state=seed)
    if name == "HistGradientBoosting":
        return HistGradientBoostingClassifier(random_state=seed)
    if name == "Stacking":
        return StackingClassifier(
            estimators=[("lr", LogisticRegression(max_iter=2000, random_state=seed)),
                        ("dt", DecisionTreeClassifier(max_depth=12, random_state=seed)),
                        ("rf", RandomForestClassifier(n_estimators=60, random_state=seed, n_jobs=-1))],
            final_estimator=LogisticRegression(max_iter=2000, random_state=seed),
            cv=3, n_jobs=1)
    raise KeyError(name)

E1_MODELS = ["LogisticRegression", "KNeighbours", "DecisionTree", "RandomForest",
                "AdaBoost", "HistGradientBoosting"]
E1_SLOW = ["GradientBoosting", "Stacking"]
E5_MODELS = ["LogisticRegression", "DecisionTree", "RandomForest", "HistGradientBoosting"]

def metrics(ytrue, ypred):
    ytrue = np.asarray(ytrue).astype(np.int8)
    ypred = np.asarray(ypred).astype(np.int8)
    tp = int(np.sum((ytrue == 1) & (ypred == 1)))
    fp = int(np.sum((ytrue == 0) & (ypred == 1)))
    fn = int(np.sum((ytrue == 1) & (ypred == 0)))
    tn = int(np.sum((ytrue == 0) & (ypred == 0)))
    prec = tp / (tp + fp) if tp + fp else 0.0
    rec = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * prec * rec / (prec + rec) if prec + rec else 0.0
    return {"precision": 100 * prec, "recall": 100 * rec, "f1": 100 * f1,
            "accuracy": 100 * (tp + tn) / max(1, tp + fp + fn + tn),
            "tp": tp, "fp": fp, "fn": fn, "tn": tn, "n_test": tp + fp + fn + tn}

def fit_eval(model_name, Xtr, ytr, Xte, yte, rep, seed=MASTER_SEED,
             balance=None, n_trees=None, return_model=False):
    Xp, yp, scaler, cw, cols = prepare(Xtr, ytr, rep, balance=balance, seed=seed)
    mdl = make_model(model_name, seed=seed, class_weight=cw, n_trees=n_trees)
    t0 = time.time()
    mdl.fit(Xp, yp)
    fit_s = time.time() - t0
    Xt = apply_prepare(Xte, cols, scaler)
    t0 = time.time()
    pred = np.empty(len(Xt), dtype=np.int8)
    STEP = 400_000
    for s in range(0, len(Xt), STEP):
        pred[s:s + STEP] = mdl.predict(Xt[s:s + STEP]).astype(np.int8)
    pred_s = time.time() - t0
    m = metrics(yte, pred)
    m.update({"model": model_name, "representation": rep, "seed": seed,
              "n_train": int(len(yp)), "train_attack_pct": float(100 * np.mean(yp)),
              "fit_seconds": round(fit_s, 2), "predict_seconds": round(pred_s, 2),
              "balance": balance or BALANCE_POLICY[rep]})
    if return_model:
        return m, mdl, scaler, cols, pred
    return m

log("Sampling, Stage D preprocessing, model zoo and metrics ready.")
log("   E1 pool allocation:", allocate(E1_POOL, CAPTURE_WEIGHTS, CH_ORDER))
log("   E5 pool allocation:", allocate(E5_TOTAL, CAPTURE_WEIGHTS, CH_ORDER))

## 4.1 Exploratory analysis

In [ ]:
EDA_SAMPLE = 120000
EDA_DIST_N = 300000

def _eda_sample(n, seed):
    alloc = allocate(n, CAPTURE_WEIGHTS, CH_ORDER)
    return build_training_set(alloc, seed)

X_eda, y_eda = _eda_sample(EDA_SAMPLE, MASTER_SEED)
log(f"EDA sample: {len(y_eda):,} frames, {100*y_eda.mean():.2f}% injected")

def _table5():
    rows = []
    ya = y_eda.astype(bool)
    for i in ENG_8:
        v = X_eda[:, i].astype(np.float64)
        rows.append({"feature": FEATURE_NAMES[i],
                     "mean": float(v.mean()), "std": float(v.std()),
                     "median": float(np.median(v)), "max": float(v.max()),
                     "mean_given_normal": float(v[~ya].mean()),
                     "mean_given_attack": float(v[ya].mean()),
                     "separation_ratio": float(v[ya].mean() / v[~ya].mean())
                                          if v[~ya].mean() else float("nan")})
    return rows

t5 = pd.DataFrame(ckpt.run("eda/table5", _table5))
save_table(t5, "table05_engineered_feature_stats",
           f"Descriptive statistics of the engineered features ({EDA_SAMPLE:,}-frame sample)")
display(t5.round(4))

def fig2():
    fig, axes = plt.subplots(1, 2, figsize=(12.4, 4.3))
    labels = [CH_PRETTY[k] for k in CH_ORDER]
    frames = np.array([VOCAB[f"ch_{k}"]["frames"] for k in CH_ORDER], dtype=float)
    atk = np.array([VOCAB[f"ch_{k}"]["attack_frames"] for k in CH_ORDER], dtype=float)
    x = np.arange(len(CH_ORDER))

    ax = axes[0]
    ax.bar(x, frames / 1e6, color="#c9d6df", edgecolor="#5b6b78",
           linewidth=0.8, label="normal + injected")
    ax.bar(x, atk / 1e6, color=C_ATK, edgecolor="#7d2f28", linewidth=0.8, label="injected")
    for xi, (fv, av) in enumerate(zip(frames, atk)):
        ax.text(xi, fv / 1e6 + 0.09, f"{fv/1e6:.2f}M", ha="center", fontsize=8.4)
        ax.text(xi, av / 1e6 / 2, f"{100*av/fv:.2f}%", ha="center", va="center",
                fontsize=8.0, color="white", fontweight="bold")
    ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=9)
    ax.set_ylabel("frames (millions)")
    ax.set_title("Capture size and injected content")
    ax.legend(fontsize=8.5, loc="upper left")

    ax = axes[1]
    nid = np.array([len(VOCAB[f"ch_{k}"]["attack_ids"]) for k in CH_ORDER], dtype=float)
    cols = [C_HI if k == "Fuzzy" else C_RAW for k in CH_ORDER]
    ax.bar(x, nid, color=cols, edgecolor="#333333", linewidth=0.8)
    for xi, v in enumerate(nid):
        ax.text(xi, v * 1.25, f"{int(v):,}", ha="center", fontsize=9, fontweight="bold")
    ax.set_yscale("log"); ax.set_ylim(0.6, max(nid) * 6)
    ax.set_xticks(x); ax.set_xticklabels(labels, fontsize=9)
    ax.set_ylabel("distinct identifiers in the attack class (log)")
    ax.set_title("The structural asymmetry this paper turns on")
    fig.tight_layout()
    return fig

save_fig(fig2(), "fig02_capture_composition")

def fig3():
    TOPN = 30
    cn = np.zeros(2048, dtype=np.int64)
    ca = np.zeros(2048, dtype=np.int64)
    for k in CH_ORDER:
        X, y = CH[k]
        STEP = 2_000_000
        for s in range(0, len(y), STEP):
            cid = np.asarray(X[s:s+STEP, 0]).astype(np.int32)
            yy = np.asarray(y[s:s+STEP]).astype(bool)
            cn += np.bincount(cid[~yy], minlength=2048)
            ca += np.bincount(cid[yy], minlength=2048)
    fig, axes = plt.subplots(1, 2, figsize=(13.0, 4.4))
    for ax, c, title, col in ((axes[0], cn, "Normal traffic", C_NORM),
                              (axes[1], ca, "Injected traffic", C_ATK)):
        idx = np.argsort(-c)[:TOPN]
        idx = idx[c[idx] > 0]
        ax.bar(np.arange(len(idx)), c[idx], color=col, edgecolor="none")
        ax.set_xticks(np.arange(len(idx)))
        ax.set_xticklabels([f"0x{i:03X}" for i in idx], rotation=90, fontsize=6.6)
        ax.set_yscale("log")
        ax.set_ylabel("frames (log)")
        ax.set_title(f"{title} - top {len(idx)} of "
                     f"{int((c>0).sum()):,} distinct identifiers")
    fig.tight_layout()
    return fig

save_fig(fig3(), "fig03_identifier_frequency")

def fig4():
    Xd, yd = _eda_sample(EDA_DIST_N, MASTER_SEED + 1)
    ya = yd.astype(bool)
    picks = [("ID_Freq", False), ("IAT_ID", True), ("Payload_Entropy", False)]
    fig, axes = plt.subplots(1, 3, figsize=(13.4, 3.9))
    for ax, (fname, logx) in zip(axes, picks):
        j = FEATURE_NAMES.index(fname)
        v = Xd[:, j].astype(np.float64)
        if logx:
            v = v[v > 0]
            yy = ya[Xd[:, j].astype(np.float64) > 0]
            bins = np.logspace(np.log10(max(v.min(), 1e-6)), np.log10(v.max()), 60)
            ax.set_xscale("log")
        else:
            yy = ya
            bins = np.linspace(v.min(), v.max(), 60)
        ax.hist(v[~yy], bins=bins, color=C_NORM, alpha=0.62, label="normal", density=True)
        ax.hist(v[yy], bins=bins, color=C_ATK, alpha=0.62, label="injected", density=True)
        ax.set_xlabel(fname); ax.set_ylabel("density")
        ax.set_title(f"{fname}\nmean {v[~yy].mean():.4g} (normal) vs {v[yy].mean():.4g} (injected)",
                     fontsize=9.4)
        ax.legend(fontsize=8.4)
    fig.suptitle(f"Class-conditional distributions of three engineered features "
                 f"({EDA_DIST_N:,}-frame sample)", y=1.06, fontsize=11)
    fig.tight_layout()
    return fig

save_fig(fig4(), "fig04_class_conditional_distributions")

def fig5():
    M = np.column_stack([X_eda.astype(np.float64), y_eda.astype(np.float64)])
    names = FEATURE_NAMES + ["label"]
    C = np.corrcoef(M, rowvar=False)
    C = np.nan_to_num(C)
    fig, ax = plt.subplots(figsize=(8.6, 7.4))
    im = ax.imshow(C, cmap="RdBu_r", vmin=-1, vmax=1)
    ax.set_xticks(range(len(names))); ax.set_xticklabels(names, rotation=90, fontsize=7.6)
    ax.set_yticks(range(len(names))); ax.set_yticklabels(names, fontsize=7.6)
    ax.grid(False)
    for i in range(len(names)):
        for j in range(len(names)):
            if abs(C[i, j]) >= 0.35:
                ax.text(j, i, f"{C[i,j]:.2f}", ha="center", va="center", fontsize=5.7,
                        color="white" if abs(C[i, j]) > 0.65 else "#222222")
    fig.colorbar(im, ax=ax, shrink=0.78, label="Pearson r")
    ax.set_title(f"Feature correlation including the binary label "
                 f"({EDA_SAMPLE:,}-frame sample)", fontsize=10.6)
    fig.tight_layout()
    save_supp(pd.DataFrame(C, index=names, columns=names).round(4),
               "table05b_feature_correlation", "Feature correlation matrix", index=True)
    return fig

save_fig(fig5(), "fig05_feature_correlation")

## 4.2 Within-vehicle evaluation (E1)

In [ ]:
E1_ALLOC = allocate(E1_POOL, CAPTURE_WEIGHTS, CH_ORDER)
log("E1 pool allocation (proportional to capture size):", E1_ALLOC)

def _e1_pool():
    Xs, ys, caps, ranks = [], [], [], []
    for ci, k in enumerate(CH_ORDER):
        m = E1_ALLOC[k]
        Xi, yi = stratified_take(k, m, MASTER_SEED * 1000 + 7 * ci + 13)
        Xs.append(Xi); ys.append(yi)
        caps.append(np.full(len(yi), ci, dtype=np.int8))
        ranks.append(np.arange(len(yi), dtype=np.int32))
    return (np.concatenate(Xs), np.concatenate(ys),
            np.concatenate(caps), np.concatenate(ranks))

X_e1, y_e1, cap_e1, rank_e1 = _e1_pool()
log(f"E1 pool: {len(y_e1):,} frames, {100*y_e1.mean():.2f}% injected")

def e1_split(protocol):
    if protocol == "random":
        tr, te = train_test_split(np.arange(len(y_e1)), test_size=0.30,
                                  stratify=y_e1, random_state=MASTER_SEED)
        return np.sort(tr), np.sort(te)
    tr, te = [], []
    for ci in range(len(CH_ORDER)):
        idx = np.flatnonzero(cap_e1 == ci)
        idx = idx[np.argsort(rank_e1[idx])]
        cut = int(round(0.70 * len(idx)))
        tr.append(idx[:cut]); te.append(idx[cut:])
    tr = np.sort(np.concatenate(tr)); te = np.sort(np.concatenate(te))
    if protocol == "chronological":
        return tr, te
    if protocol == "prevalence_matched":
        target = float(y_e1.mean())
        yt = y_e1[te]
        a = te[yt == 1]; n = te[yt == 0]
        if len(a) / len(te) < target:
            keep_n = int(round(len(a) * (1 - target) / target))
            rs = np.random.RandomState(MASTER_SEED)
            n = rs.choice(n, size=min(keep_n, len(n)), replace=False)
        else:
            keep_a = int(round(len(n) * target / (1 - target)))
            rs = np.random.RandomState(MASTER_SEED)
            a = rs.choice(a, size=min(keep_a, len(a)), replace=False)
        return tr, np.sort(np.concatenate([a, n]))
    raise KeyError(protocol)

E1_PROTOCOLS = ["random", "chronological", "prevalence_matched"]
E1_SPLITS = {}
for p in E1_PROTOCOLS:
    tr, te = e1_split(p)
    E1_SPLITS[p] = (tr, te)
    log(f"   {p:20s} train={len(tr):,} ({100*y_e1[tr].mean():.2f}% attack)   "
        f"test={len(te):,} ({100*y_e1[te].mean():.2f}% attack)")

def _e1_split_prevalence():
    return [{"protocol": p,
             "train_frames": int(len(E1_SPLITS[p][0])),
             "train_attack_pct": float(100 * y_e1[E1_SPLITS[p][0]].mean()),
             "test_frames": int(len(E1_SPLITS[p][1])),
             "test_attack_pct": float(100 * y_e1[E1_SPLITS[p][1]].mean())}
            for p in E1_PROTOCOLS]

t6b = pd.DataFrame(ckpt.run("E1/split_prevalence", _e1_split_prevalence))
save_supp(t6b, "table06b_split_prevalence",
           "Attack prevalence of each within-vehicle split partition")
display(t6b.round(2))

models_e1 = E1_MODELS + (E1_SLOW if RUN_SLOW_E1 else [])
E1_RESULTS = []
for protocol in E1_PROTOCOLS:
    tr, te = E1_SPLITS[protocol]
    for rep in ["raw10", "all18"]:
        for mname in models_e1:
            key = f"E1/split/{protocol}/{rep}/{mname}"
            r = ckpt.run(key, lambda mname=mname, rep=rep, tr=tr, te=te: fit_eval(
                mname, X_e1[tr], y_e1[tr], X_e1[te], y_e1[te], rep,
                n_trees=RF_E1_TREES))
            r["protocol"] = protocol
            E1_RESULTS.append(r)

e1_df = pd.DataFrame(E1_RESULTS)
e1_wide = e1_df.pivot_table(index=["protocol", "model"], columns="representation",
                            values="f1").reset_index()
e1_wide.columns.name = None
if "all18" in e1_wide and "raw10" in e1_wide:
    e1_wide["delta_all18_minus_raw10"] = e1_wide["all18"] - e1_wide["raw10"]
save_supp(e1_df, "table06a_within_vehicle_splits_full",
           "Within-vehicle results, all three split protocols (full metrics)")
save_supp(e1_wide, "table06c_within_vehicle_splits_f1",
           "Within-vehicle F1 by split protocol and representation")
display(e1_wide.round(2))

In [ ]:
E1_CV_ALLOC = allocate(E1_CV_POOL, CAPTURE_WEIGHTS, CH_ORDER)
X_cv, y_cv = build_training_set(E1_CV_ALLOC, MASTER_SEED)
log(f"E1 CV pool: {len(y_cv):,} frames, {100*y_cv.mean():.2f}% injected")

def cv_f1(mname, rep):
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=MASTER_SEED)
    scores, folds = [], []
    for fi, (tr, te) in enumerate(skf.split(X_cv, y_cv)):
        m = fit_eval(mname, X_cv[tr], y_cv[tr], X_cv[te], y_cv[te], rep,
                     n_trees=RF_E1_TREES)
        scores.append(m["f1"])
        folds.append({"fold": fi, **{k: m[k] for k in
                                     ("precision", "recall", "f1", "tp", "fp", "fn", "tn")}})
    return {"model": mname, "representation": rep,
            "f1_mean": float(np.mean(scores)), "f1_std": float(np.std(scores)),
            "folds": folds}

CV = {}
for mname in E1_MODELS:
    for rep in ["raw10", "all18"]:
        CV[(mname, rep)] = ckpt.run(f"E1/cv/{rep}/{mname}",
                                    lambda mname=mname, rep=rep: cv_f1(mname, rep))

def _table6():
    rows = []
    for mname in E1_MODELS:
        r, c = CV[(mname, "raw10")], CV[(mname, "all18")]
        rows.append({"model": mname,
                     "raw_f1_mean": r["f1_mean"], "raw_f1_std": r["f1_std"],
                     "canshield_f1_mean": c["f1_mean"], "canshield_f1_std": c["f1_std"],
                     "delta": c["f1_mean"] - r["f1_mean"]})
    return rows

t6 = pd.DataFrame(ckpt.run("tables/table6", _table6))
t6["raw_features"] = t6.apply(lambda r: f"{r.raw_f1_mean:.2f} +/- {r.raw_f1_std:.2f}", axis=1)
t6["CAN_SHIELD"] = t6.apply(lambda r: f"{r.canshield_f1_mean:.2f} +/- {r.canshield_f1_std:.2f}", axis=1)
save_table(t6, "table06_within_vehicle_cv",
           "Within-vehicle F1, five-fold stratified cross-validation (per cent)")
display(t6[["model", "raw_features", "CAN_SHIELD", "delta"]].round(2))

def _e1_wilcoxon():
    raw = np.array([CV[(m, "raw10")]["f1_mean"] for m in E1_MODELS])
    cs = np.array([CV[(m, "all18")]["f1_mean"] for m in E1_MODELS])
    d = cs - raw
    out = {"n": len(E1_MODELS), "mean_delta": float(d.mean()),
           "models": E1_MODELS, "raw_means": raw.tolist(), "canshield_means": cs.tolist()}
    try:
        st, p = stats.wilcoxon(cs, raw, alternative="greater", zero_method="wilcox")
        out["W"] = float(st); out["p_one_sided"] = float(p)
    except Exception as e:
        out["W"] = None; out["p_one_sided"] = None; out["error"] = str(e)
    return out

W1 = ckpt.run("E1/wilcoxon_cv", _e1_wilcoxon)
log(f"E1 Wilcoxon signed-rank (CAN-SHIELD > raw, one-sided, n={W1['n']}): "
    f"W = {W1['W']}, p = {W1['p_one_sided']}")
log(f"   mean change across the {W1['n']} models: {W1['mean_delta']:+.2f} percentage points")
save_supp(pd.DataFrame([W1])[["n", "mean_delta", "W", "p_one_sided"]],
           "table06d_within_vehicle_wilcoxon",
           "Wilcoxon signed-rank test over the paired within-vehicle CV means")

In [ ]:
def fig6():
    x = np.arange(len(E1_MODELS)); w = 0.38
    raw_m = [CV[(m, "raw10")]["f1_mean"] for m in E1_MODELS]
    raw_s = [CV[(m, "raw10")]["f1_std"]  for m in E1_MODELS]
    cs_m = [CV[(m, "all18")]["f1_mean"] for m in E1_MODELS]
    cs_s = [CV[(m, "all18")]["f1_std"]  for m in E1_MODELS]
    fig, ax = plt.subplots(figsize=(10.4, 4.6))
    ax.bar(x - w/2, raw_m, w, yerr=raw_s, capsize=3, color=C_RAW,
           edgecolor="#4a4f54", linewidth=0.8, label="Raw 10")
    ax.bar(x + w/2, cs_m, w, yerr=cs_s, capsize=3, color=C_ENG,
           edgecolor="#0f514d", linewidth=0.8, label="CAN-SHIELD 18")
    lo = max(0.0, min(raw_m + cs_m) - 6)
    ax.set_ylim(lo, 101.5)
    for xi, v in zip(x - w/2, raw_m): ax.text(xi, v + 0.5, f"{v:.2f}", ha="center", fontsize=7.2)
    for xi, v in zip(x + w/2, cs_m):  ax.text(xi, v + 0.5, f"{v:.2f}", ha="center", fontsize=7.2)
    ax.set_xticks(x); ax.set_xticklabels(E1_MODELS, rotation=18, ha="right", fontsize=8.6)
    ax.set_ylabel("F1 (%)")
    ax.set_title("Within-vehicle five-fold cross-validated F1 (error bars: one standard deviation)")
    ax.legend(fontsize=8.8, loc="lower right")
    fig.tight_layout()
    return fig

save_fig(fig6(), "fig06_within_vehicle_cv_f1")

## 4.3-4.5 Cross-vehicle transfer (E2, E3, E4)

In [ ]:
E2_COMPOSITIONS = {
    "flooding_only":      {"DoS": E2_TOTAL},
    "flooding_gear_rpm":  {"DoS": E2_TOTAL // 3, "gear": E2_TOTAL // 3, "RPM": E2_TOTAL // 3},
    "all_four_with_fuzzing": {k: E2_TOTAL // 4 for k in CH_ORDER},
}
E2_LABEL = {"flooding_only": "Flooding only",
            "flooding_gear_rpm": "Flooding + gear + RPM spoofing",
            "all_four_with_fuzzing": "All four, fuzzing included"}

def run_transfer(spec, rep, seed, balance=None, tag="", dedup=False):
    Xtr, ytr = build_training_set(spec, seed, dedup=dedup)
    nid = train_attack_id_count(Xtr, ytr)
    m = fit_eval("RandomForest", Xtr, ytr, X_OT, y_OT, rep, seed=seed,
                 balance=balance, n_trees=RF_TRANSFER_TREES)
    m.update({"training_attack_ids": nid, "composition": tag,
              "spec": {k: int(v) for k, v in spec.items()}})
    del Xtr, ytr
    gc.collect()
    return m

E2 = {}
for comp, spec in E2_COMPOSITIONS.items():
    for rep in ["raw10", "eng8"]:
        E2[(comp, rep)] = ckpt.run(
            f"E2/{comp}/{rep}",
            lambda spec=spec, rep=rep, comp=comp: run_transfer(
                spec, rep, MASTER_SEED, tag=comp))

e2_rows = []
for comp in E2_COMPOSITIONS:
    r, e = E2[(comp, "raw10")], E2[(comp, "eng8")]
    e2_rows.append({"training_attack_classes": E2_LABEL[comp],
                    "attack_ids_in_training": r["training_attack_ids"],
                    "raw_f1": r["f1"], "raw_precision": r["precision"], "raw_recall": r["recall"],
                    "raw_false_positives": r["fp"],
                    "engineered_f1": e["f1"], "engineered_precision": e["precision"],
                    "engineered_recall": e["recall"], "engineered_false_positives": e["fp"]})
e2_df = pd.DataFrame(e2_rows)
display(e2_df.round(2))

def fig7():
    labels = [E2_LABEL[c] for c in E2_COMPOSITIONS]
    raw = [E2[(c, "raw10")]["f1"] for c in E2_COMPOSITIONS]
    eng = [E2[(c, "eng8")]["f1"]  for c in E2_COMPOSITIONS]
    nid = [E2[(c, "raw10")]["training_attack_ids"] for c in E2_COMPOSITIONS]
    x = np.arange(len(labels)); w = 0.38
    fig, ax = plt.subplots(figsize=(10.0, 4.8))
    ax.bar(x - w/2, raw, w, color=C_RAW, edgecolor="#4a4f54", linewidth=0.8, label="Raw 10")
    ax.bar(x + w/2, eng, w, color=C_ENG, edgecolor="#0f514d", linewidth=0.8, label="Engineered 8")
    for xi, v in zip(x - w/2, raw): ax.text(xi, v + 1.2, f"{v:.2f}", ha="center", fontsize=8.4)
    for xi, v in zip(x + w/2, eng): ax.text(xi, v + 1.2, f"{v:.2f}", ha="center", fontsize=8.4)
    ax.set_xticks(x)
    ax.set_xticklabels([f"{l}\n({n:,} attack identifiers)" for l, n in zip(labels, nid)],
                       fontsize=8.6)
    ax.set_ylim(0, 108); ax.set_ylabel("cross-vehicle F1 (%)")
    ax.set_title("Cross-vehicle F1 by training-set attack composition\n"
                 f"training set held at {E2_TOTAL:,} frames in every bar; "
                 f"test set {OTIDS_TEST['frames']:,} KIA Soul frames", fontsize=10.4)
    ax.legend(fontsize=9)
    fig.tight_layout()
    return fig

save_fig(fig7(), "fig07_composition_ablation")

In [ ]:
def _otids_training_pool():
    Xd, yd = OT["dos"]
    yd = np.asarray(yd)
    ai = np.flatnonzero(yd == 1); ni = np.flatnonzero(yd == 0)
    mt = int(round(E2_TOTAL * len(ai) / len(yd)))
    rs = np.random.RandomState(MASTER_SEED)
    sel = np.sort(np.concatenate([rs.choice(ai, mt, replace=False),
                                  rs.choice(ni, E2_TOTAL - mt, replace=False)]))
    return {"index": sel.astype(np.int64)}

_e4idx = ckpt.run_np("E4/training_index", _otids_training_pool)["index"]

def run_e4(rep):
    Xd, yd = OT["dos"]
    Xtr = np.asarray(Xd)[_e4idx]; ytr = np.asarray(yd)[_e4idx]
    Xte, yte = CH["DoS"]
    m, mdl, scaler, cols, pred = fit_eval(
        "RandomForest", Xtr, ytr, Xte, yte, rep, seed=MASTER_SEED,
        n_trees=RF_TRANSFER_TREES, return_model=True)
    m["training_attack_ids"] = train_attack_id_count(Xtr, ytr)
    m["composition"] = "otids_flooding_only"
    yte = np.asarray(yte)
    cid = np.asarray(Xte[:, 0]).astype(np.int32)
    fp_mask = (yte == 0) & (pred == 1)
    fp_by_id = np.bincount(cid[fp_mask], minlength=2048)
    norm_by_id = np.bincount(cid[yte == 0], minlength=2048)
    ot_known = set(VOCAB["ot_attack_free"]["normal_ids"])
    rows = []
    for i in np.flatnonzero(norm_by_id):
        rows.append({"can_id": int(i), "hex": f"0x{i:03X}",
                     "normal_frames": int(norm_by_id[i]),
                     "false_positives": int(fp_by_id[i]),
                     "fp_rate_pct": float(100 * fp_by_id[i] / norm_by_id[i]),
                     "familiar_to_otids_model": bool(i in ot_known)})
    m["fp_attribution"] = sorted(rows, key=lambda r: -r["false_positives"])
    return m

E4 = {}
for rep in ["raw10", "eng8"]:
    E4[rep] = ckpt.run(f"E4/{rep}", lambda rep=rep: run_e4(rep))

def _table7():
    rows = list(e2_rows)
    r, e = E4["raw10"], E4["eng8"]
    rows.append({"training_attack_classes": "Reverse direction: KIA Soul -> Sonata, flooding only",
                 "attack_ids_in_training": r["training_attack_ids"],
                 "raw_f1": r["f1"], "raw_precision": r["precision"], "raw_recall": r["recall"],
                 "raw_false_positives": r["fp"],
                 "engineered_f1": e["f1"], "engineered_precision": e["precision"],
                 "engineered_recall": e["recall"], "engineered_false_positives": e["fp"]})
    return rows

t7 = pd.DataFrame(ckpt.run("tables/table7", _table7))
save_table(t7, "table07_cross_vehicle_transfer",
           "Cross-vehicle transfer by training-set composition (Random Forest)", float_fmt="%.2f")
display(t7.round(2))

fp_df = pd.DataFrame(E4["raw10"]["fp_attribution"])
fp_top = fp_df.head(12)
save_supp(fp_df, "tableA4_e4_false_positive_attribution",
           "Reverse transfer: false positives attributed to arbitration identifier")
log("Reverse transfer (raw 10): where do the false positives come from?")
log(f"   total false positives        : {E4['raw10']['fp']:,}")
unfam = fp_df[~fp_df["familiar_to_otids_model"]]
log(f"   legitimate identifiers in the test capture        : {len(fp_df)}")
log(f"   of those, unfamiliar to the OTIDS-trained model   : {len(unfam)}")
log(f"   normal frames carried by the unfamiliar ones      : {int(unfam['normal_frames'].sum()):,}")
log(f"   false positives falling on the unfamiliar ones    : {int(unfam['false_positives'].sum()):,} "
    f"({100*unfam['false_positives'].sum()/max(1,E4['raw10']['fp']):.1f}% of all FPs)")
display(fp_top.round(2))

In [ ]:
E3_BASE = {"DoS": E3_BASE_EACH, "gear": E3_BASE_EACH, "RPM": E3_BASE_EACH}

def e3_spec(pct):
    spec = dict(E3_BASE)
    add = int(round(E3_FUZZ_SUB * pct / 100.0))
    if add:
        spec["Fuzzy"] = add
    return spec

E3 = {}
for pct in E3_DOSES:
    for rep in ["raw10", "eng8"]:
        E3[(pct, rep)] = ckpt.run(
            f"E3/dose{pct:03d}/{rep}",
            lambda pct=pct, rep=rep: run_transfer(e3_spec(pct), rep, MASTER_SEED,
                                                  tag=f"dose_{pct}pct"))

def e3_key(pct, rep, sd):
    if pct == 0:
        return f"E3b/no_fuzzing/{rep}/seed{sd}"
    if pct == 100:
        return f"E3b/with_fuzzing/{rep}/seed{sd}"
    return f"E3/dose{pct:03d}/{rep}/seed{sd}"

E3MS = {}
for sd in E3_SEEDS:
    for pct in E3_DOSES:
        for rep in ["raw10", "eng8"]:
            E3MS[(pct, rep, sd)] = ckpt.run(
                e3_key(pct, rep, sd),
                lambda pct=pct, rep=rep, sd=sd: run_transfer(
                    e3_spec(pct), rep, sd, tag=f"dose_{pct}pct_seed{sd}"))

def _agg(pct, rep, metric):
    v = np.array([E3MS[(pct, rep, s)][metric] for s in E3_SEEDS], dtype=float)
    return float(v.mean()), float(v.std())

def _table8():
    rows = []
    for pct in E3_DOSES:
        add = int(round(E3_FUZZ_SUB * pct / 100.0))
        ids = np.array([E3MS[(pct, "raw10", s)]["training_attack_ids"] for s in E3_SEEDS])
        rf1, rf1s = _agg(pct, "raw10", "f1")
        rpr, rprs = _agg(pct, "raw10", "precision")
        rrc, _    = _agg(pct, "raw10", "recall")
        rfp, _    = _agg(pct, "raw10", "fp")
        ef1, ef1s = _agg(pct, "eng8", "f1")
        epr, eprs = _agg(pct, "eng8", "precision")
        rows.append({"fuzzing_pct_of_subsample": pct,
                     "fuzzing_frames_added": add,
                     "n_seeds": len(E3_SEEDS),
                     "distinct_attack_ids_mean": float(ids.mean()),
                     "distinct_attack_ids_min": int(ids.min()),
                     "distinct_attack_ids_max": int(ids.max()),
                     "train_frames": E3MS[(pct, "raw10", E3_SEEDS[0])]["n_train"],
                     "raw_f1_mean": rf1, "raw_f1_std": rf1s,
                     "raw_precision_mean": rpr, "raw_precision_std": rprs,
                     "raw_recall_mean": rrc, "raw_false_positives_mean": rfp,
                     "eng8_f1_mean": ef1, "eng8_f1_std": ef1s,
                     "eng8_precision_mean": epr, "eng8_precision_std": eprs,
                     "raw_f1_seed42": E3[(pct, "raw10")]["f1"],
                     "eng8_f1_seed42": E3[(pct, "eng8")]["f1"],
                     "distinct_attack_ids_seed42": E3[(pct, "raw10")]["training_attack_ids"]})
    return rows

t8 = pd.DataFrame(ckpt.run("tables/table8_v2", _table8))
t8["raw_F1"] = t8.apply(lambda r: f"{r.raw_f1_mean:.2f} +/- {r.raw_f1_std:.2f}", axis=1)
t8["raw_precision"] = t8.apply(lambda r: f"{r.raw_precision_mean:.2f} +/- {r.raw_precision_std:.2f}", axis=1)
t8["eng8_F1"] = t8.apply(lambda r: f"{r.eng8_f1_mean:.2f} +/- {r.eng8_f1_std:.2f}", axis=1)
t8["eng8_precision"] = t8.apply(lambda r: f"{r.eng8_precision_mean:.2f} +/- {r.eng8_precision_std:.2f}", axis=1)
save_table(t8, "table08_dose_response",
           f"Dose-response of cross-vehicle transfer, mean +/- s.d. over {len(E3_SEEDS)} draws",
           float_fmt="%.2f")
display(t8[["fuzzing_frames_added", "distinct_attack_ids_mean", "raw_F1", "raw_precision",
            "eng8_F1", "eng8_precision", "raw_f1_seed42"]].round(2))

save_supp(pd.DataFrame([{"seed": s, "fuzzing_frames_added": int(round(E3_FUZZ_SUB * p / 100.0)),
                         "representation": REP_PRETTY[r],
                         "f1": E3MS[(p, r, s)]["f1"], "precision": E3MS[(p, r, s)]["precision"],
                         "recall": E3MS[(p, r, s)]["recall"], "fp": E3MS[(p, r, s)]["fp"],
                         "training_attack_ids": E3MS[(p, r, s)]["training_attack_ids"]}
                        for s in E3_SEEDS for p in E3_DOSES for r in ("raw10", "eng8")]),
          "table08b_dose_response_per_seed", "Dose-response, every seed", float_fmt="%.2f")
save_supp(pd.DataFrame([{"fuzzing_frames_added": int(round(E3_FUZZ_SUB * p / 100.0)),
                         "distinct_attack_ids": E3[(p, "raw10")]["training_attack_ids"],
                         "raw_f1": E3[(p, "raw10")]["f1"], "raw_precision": E3[(p, "raw10")]["precision"],
                         "eng8_f1": E3[(p, "eng8")]["f1"], "eng8_precision": E3[(p, "eng8")]["precision"]}
                        for p in E3_DOSES]),
          "table08s_dose_response_seed42", "Dose-response, single draw (seed 42)",
          float_fmt="%.2f")

def _e3_trend():
    doses = np.array(E3_DOSES, dtype=float)
    R = np.array([[E3MS[(p, "raw10", s)]["f1"] for p in E3_DOSES] for s in E3_SEEDS])
    E = np.array([[E3MS[(p, "eng8", s)]["f1"]  for p in E3_DOSES] for s in E3_SEEDS])
    out = {"n_seeds": int(R.shape[0]), "n_doses": int(R.shape[1])}
    for tag, M in (("raw", R), ("eng8", E)):
        try:
            res = stats.page_trend_test(-M)
            out[f"{tag}_page_L"] = float(res.statistic)
            out[f"{tag}_page_p"] = float(res.pvalue)
            out[f"{tag}_page_method"] = str(res.method)
        except Exception as e:
            out[f"{tag}_page_error"] = str(e)
        rho, p = stats.spearmanr(np.tile(doses, M.shape[0]), M.ravel())
        out[f"{tag}_spearman_rho_pooled"] = float(rho)
        out[f"{tag}_spearman_p_pooled"] = float(p)
    mean_curve = R.mean(axis=0)
    sd_curve = R.std(axis=0)
    rises = [{"from_frames": int(round(E3_FUZZ_SUB * E3_DOSES[i] / 100.0)),
              "to_frames": int(round(E3_FUZZ_SUB * E3_DOSES[i + 1] / 100.0)),
              "rise": float(mean_curve[i + 1] - mean_curve[i]),
              "pooled_sd": float(np.sqrt((sd_curve[i] ** 2 + sd_curve[i + 1] ** 2) / 2))}
             for i in range(len(E3_DOSES) - 1) if mean_curve[i + 1] > mean_curve[i] + 1e-9]
    out["raw_mean_curve"] = mean_curve.tolist()
    out["raw_sd_curve"] = sd_curve.tolist()
    out["raw_mean_curve_monotonic_non_increasing"] = bool(np.all(np.diff(mean_curve) <= 1e-9))
    out["raw_mean_curve_rises"] = rises
    out["seeds_with_monotonic_curve"] = int(sum(np.all(np.diff(r) <= 1e-9) for r in R))
    out["first_dose_cost_mean"] = float(mean_curve[0] - mean_curve[1])
    out["first_dose_cost_sd"] = float(np.std(R[:, 0] - R[:, 1]))
    out["eng8_mean_min"] = float(E.mean(axis=0).min())
    out["eng8_mean_max"] = float(E.mean(axis=0).max())
    return out

TREND = ckpt.run("E3/trend_tests_v2", _e3_trend)
log(f"Dose-response over {TREND['n_seeds']} seeds x {TREND['n_doses']} doses:")
if "raw_page_p" in TREND:
    log(f"   Page's L trend test (raw F1 falls with dose): L = {TREND['raw_page_L']:.1f}, "
        f"p = {TREND['raw_page_p']:.2e} ({TREND['raw_page_method']})")
log(f"   Spearman rho, dose vs raw F1 (pooled)      : {TREND['raw_spearman_rho_pooled']:+.3f} "
    f"(p = {TREND['raw_spearman_p_pooled']:.2e})")
log(f"   mean curve monotonic non-increasing        : {TREND['raw_mean_curve_monotonic_non_increasing']}")
for r in TREND["raw_mean_curve_rises"]:
    log(f"      mean rises {r['rise']:+.2f} between {r['from_frames']:,} and {r['to_frames']:,} frames "
        f"(pooled s.d. {r['pooled_sd']:.2f})")
log(f"   seeds whose own curve is monotonic         : {TREND['seeds_with_monotonic_curve']} of {TREND['n_seeds']}")
log(f"   smallest dose cost                         : {TREND['first_dose_cost_mean']:.2f} "
    f"+/- {TREND['first_dose_cost_sd']:.2f} F1 points")
log(f"   engineered-8 mean range                    : {TREND['eng8_mean_min']:.2f} - {TREND['eng8_mean_max']:.2f}")
if "eng8_page_p" in TREND:
    log(f"   engineered-8 trend (Page's L)              : p = {TREND['eng8_page_p']:.3f}")
save_supp(pd.DataFrame([{k: v for k, v in TREND.items()
                         if not isinstance(v, (list, dict))}]),
          "table08c_dose_trend_tests", "Trend tests for the dose-response sweep", float_fmt="%.4g")

def fig9():
    fig, axes = plt.subplots(1, 2, figsize=(12.6, 4.5))
    xf = t8["fuzzing_frames_added"].to_numpy()
    xi = t8["distinct_attack_ids_mean"].to_numpy()
    for ax, xv, xlab in ((axes[0], xf, "fuzzing frames added to the training set"),
                         (axes[1], xi, "distinct attack identifiers in the training set (mean)")):
        xp = np.where(xv == 0, 0.5, xv).astype(float)
        for s in E3_SEEDS:
            ax.plot(xp, [E3MS[(p, "raw10", s)]["f1"] for p in E3_DOSES], "-",
                    color=C_RAW, lw=0.7, alpha=0.35)
        ax.errorbar(xp, t8["raw_f1_mean"], yerr=t8["raw_f1_std"], fmt="o-", color=C_RAW,
                    mec="#3f4448", lw=1.8, ms=6, capsize=3, label="Raw 10 F1")
        ax.errorbar(xp, t8["raw_precision_mean"], yerr=t8["raw_precision_std"], fmt="s--",
                    color=C_HI, mec="#8a4a1d", lw=1.4, ms=5, capsize=3, label="Raw 10 precision")
        ax.errorbar(xp, t8["eng8_f1_mean"], yerr=t8["eng8_f1_std"], fmt="o-", color=C_ENG,
                    mec="#0f514d", lw=1.8, ms=6, capsize=3, label="Engineered 8 F1")
        ax.set_xscale("log")
        ax.set_xticks(xp)
        ax.set_xticklabels([("0" if v == 0 else f"{int(round(v)):,}") for v in xv],
                           rotation=45, fontsize=7.8)
        ax.minorticks_off()
        ax.set_xlabel(xlab); ax.set_ylabel("per cent"); ax.set_ylim(0, 105)
        ax.legend(fontsize=8.2, loc="lower left")
    axes[0].set_title("Transfer against fuzzing dose")
    axes[1].set_title("The same curve against identifier novelty")
    sub = (f"mean +/- s.d. over {len(E3_SEEDS)} draws; thin lines are individual draws")
    if "raw_page_p" in TREND:
        sub += f"; Page's L trend test p = {TREND['raw_page_p']:.1e}"
    fig.suptitle("Dose-response of cross-vehicle transfer\n" + sub, y=1.06, fontsize=10.6)
    fig.tight_layout()
    return fig

save_fig(fig9(), "fig09_dose_response")

In [ ]:
E3B_CONDITIONS = {"no_fuzzing": e3_spec(0), "with_fuzzing": e3_spec(100)}

E3B = {}
for cond, spec in E3B_CONDITIONS.items():
    for rep in ["raw10", "eng8"]:
        for sd in MULTI_SEEDS:
            E3B[(cond, rep, sd)] = ckpt.run(
                f"E3b/{cond}/{rep}/seed{sd}",
                lambda spec=spec, rep=rep, sd=sd, cond=cond: run_transfer(
                    spec, rep, sd, tag=f"{cond}_seed{sd}"))

def _e3b_summary():
    rows = []
    for cond in E3B_CONDITIONS:
        for rep in ["raw10", "eng8"]:
            f1 = np.array([E3B[(cond, rep, s)]["f1"] for s in MULTI_SEEDS])
            pr = np.array([E3B[(cond, rep, s)]["precision"] for s in MULTI_SEEDS])
            rc = np.array([E3B[(cond, rep, s)]["recall"] for s in MULTI_SEEDS])
            rows.append({"condition": cond, "representation": rep, "n_draws": len(MULTI_SEEDS),
                         "f1_mean": f1.mean(), "f1_std": f1.std(),
                         "precision_mean": pr.mean(), "precision_std": pr.std(),
                         "recall_mean": rc.mean(), "recall_std": rc.std(),
                         "f1_per_draw": f1.tolist()})
    return rows

t_e3b = pd.DataFrame(ckpt.run("tables/table7b_five_draws", _e3b_summary))
save_supp(t_e3b.drop(columns=["f1_per_draw"]), "table07b_five_draws",
           "Cross-vehicle transfer over five independent training draws", float_fmt="%.2f")
display(t_e3b.drop(columns=["f1_per_draw"]).round(2))

def _e3b_wilcoxon():
    out = {}
    for rep in ["raw10", "eng8"]:
        a = np.array([E3B[("no_fuzzing", rep, s)]["f1"] for s in MULTI_SEEDS])
        b = np.array([E3B[("with_fuzzing", rep, s)]["f1"] for s in MULTI_SEEDS])
        d = {"n": len(MULTI_SEEDS), "no_fuzzing_f1": a.tolist(), "with_fuzzing_f1": b.tolist(),
             "mean_drop": float(a.mean() - b.mean())}
        try:
            st, p = stats.wilcoxon(a, b, alternative="greater", zero_method="wilcox")
            d["W"] = float(st); d["p_one_sided"] = float(p)
        except Exception as e:
            d["W"] = None; d["p_one_sided"] = None; d["error"] = str(e)
        out[rep] = d
    return out

W2 = ckpt.run("E3b/wilcoxon", _e3b_wilcoxon)
for rep, d in W2.items():
    log(f"E3b Wilcoxon ({REP_PRETTY[rep]}, no-fuzzing > with-fuzzing, one-sided, n={d['n']}): "
        f"W = {d['W']}, p = {d['p_one_sided']}, mean drop = {d['mean_drop']:.2f} F1 points")
save_supp(pd.DataFrame([{"representation": r, **{k: v for k, v in d.items()
                                                  if k not in ("no_fuzzing_f1", "with_fuzzing_f1")}}
                         for r, d in W2.items()]),
           "table07c_five_draws_wilcoxon", "Wilcoxon signed-rank over the five paired draws")

def fig8():
    conds = list(E3B_CONDITIONS)
    labels = {"no_fuzzing": "No fuzzing in training", "with_fuzzing": "Fuzzing in training"}
    x = np.arange(len(conds)); w = 0.36
    fig, axes = plt.subplots(1, 2, figsize=(12.0, 4.4))
    for ax, metric, title in ((axes[0], "f1", "F1"), (axes[1], "precision", "Precision")):
        for off, rep, col, ec in ((-w/2, "raw10", C_RAW, "#4a4f54"),
                                  (+w/2, "eng8", C_ENG, "#0f514d")):
            mu = [np.mean([E3B[(c, rep, s)][metric] for s in MULTI_SEEDS]) for c in conds]
            sd = [np.std([E3B[(c, rep, s)][metric] for s in MULTI_SEEDS]) for c in conds]
            ax.bar(x + off, mu, w, yerr=sd, capsize=4, color=col, edgecolor=ec,
                   linewidth=0.8, label=REP_PRETTY[rep])
            for xi, m, s in zip(x + off, mu, sd):
                ax.text(xi, m + s + 1.6, f"{m:.2f}\n+/-{s:.2f}", ha="center", fontsize=7.4)
        ax.set_xticks(x); ax.set_xticklabels([labels[c] for c in conds], fontsize=9)
        ax.set_ylim(0, 115); ax.set_ylabel(f"{title} (%)")
        ax.set_title(f"Cross-vehicle {title}, mean +/- s.d. over {len(MULTI_SEEDS)} draws")
        ax.legend(fontsize=8.6, loc="lower left")
    fig.tight_layout()
    return fig

save_fig(fig8(), "fig08_five_independent_draws")

In [ ]:
BAL_METHODS = {"manuscript": None, "class_weight": "class_weight"}

BAL = {}
for comp, spec in E2_COMPOSITIONS.items():
    for rep in ["raw10", "eng8"]:
        for mlabel, mval in BAL_METHODS.items():
            if mlabel == "manuscript":
                BAL[(comp, rep, mlabel)] = E2[(comp, rep)]
                continue
            BAL[(comp, rep, mlabel)] = ckpt.run(
                f"BAL/{comp}/{rep}/{mlabel}",
                lambda spec=spec, rep=rep, mval=mval, comp=comp: run_transfer(
                    spec, rep, MASTER_SEED, balance=mval, tag=comp))

for mlabel, mval in BAL_METHODS.items():
    key = f"BAL/within_vehicle_all18/{mlabel}"
    BAL[("within_all18", "all18", mlabel)] = ckpt.run(
        key, lambda mval=mval: fit_eval(
            "RandomForest", X_e1[E1_SPLITS["random"][0]], y_e1[E1_SPLITS["random"][0]],
            X_e1[E1_SPLITS["random"][1]], y_e1[E1_SPLITS["random"][1]], "all18",
            balance=mval if mval else "smote", n_trees=RF_E1_TREES))

def _table_bal():
    rows = []
    for comp in E2_COMPOSITIONS:
        for rep in ["raw10", "eng8"]:
            a = BAL[(comp, rep, "manuscript")]
            b = BAL[(comp, rep, "class_weight")]
            rows.append({"condition": E2_LABEL[comp], "representation": REP_PRETTY[rep],
                         "manuscript_protocol_f1": a["f1"], "class_weight_f1": b["f1"],
                         "difference": a["f1"] - b["f1"]})
    a = BAL[("within_all18", "all18", "manuscript")]
    b = BAL[("within_all18", "all18", "class_weight")]
    rows.append({"condition": "Within-vehicle (SMOTE does interpolate the identifier here)",
                 "representation": REP_PRETTY["all18"],
                 "manuscript_protocol_f1": a["f1"], "class_weight_f1": b["f1"],
                 "difference": a["f1"] - b["f1"]})
    return rows

t_bal = pd.DataFrame(ckpt.run("tables/balancing_control", _table_bal))
save_supp(t_bal, "table07d_balancing_control",
           "Composition ablation repeated with class weighting in place of SMOTE", float_fmt="%.2f")
display(t_bal.round(2))
log("Largest manuscript-vs-class-weighting difference: "
    f"{t_bal['difference'].abs().max():.2f} F1 points")

In [ ]:
E5_ALLOC = allocate(E5_TOTAL, CAPTURE_WEIGHTS, CH_ORDER)
log("E5 allocation (proportional to capture size, all four attacks):", E5_ALLOC)

def run_e5(mname, rep, seed=MASTER_SEED):
    Xtr, ytr = build_training_set(E5_ALLOC, seed)
    m = fit_eval(mname, Xtr, ytr, X_OT, y_OT, rep, seed=seed,
                 n_trees=RF_TRANSFER_TREES)
    m["training_attack_ids"] = train_attack_id_count(Xtr, ytr)
    del Xtr, ytr
    gc.collect()
    return m

E5 = {}
for mname in E5_MODELS:
    for rep in ["raw10", "all18", "eng8"]:
        E5[(mname, rep)] = ckpt.run(f"E5/{mname}/{rep}",
                                    lambda mname=mname, rep=rep: run_e5(mname, rep))

E5MS = {}
for sd in E5_SEEDS:
    for mname in E5_MODELS:
        for rep in ["raw10", "all18", "eng8"]:
            E5MS[(mname, rep, sd)] = ckpt.run(
                f"E5/{mname}/{rep}/seed{sd}",
                lambda mname=mname, rep=rep, sd=sd: run_e5(mname, rep, sd))

def _table9():
    rows = []
    for mname in E5_MODELS:
        for rep in ["raw10", "all18", "eng8"]:
            v = {k: np.array([E5MS[(mname, rep, s)][k] for s in E5_SEEDS], dtype=float)
                 for k in ("precision", "recall", "f1", "fp")}
            rows.append({"model": mname, "representation": REP_PRETTY[rep],
                         "n_seeds": len(E5_SEEDS),
                         "precision_mean": v["precision"].mean(), "precision_std": v["precision"].std(),
                         "recall_mean": v["recall"].mean(),
                         "f1_mean": v["f1"].mean(), "f1_std": v["f1"].std(),
                         "f1_min": v["f1"].min(), "f1_max": v["f1"].max(),
                         "false_positives_mean": v["fp"].mean(),
                         "f1_seed42": E5[(mname, rep)]["f1"],
                         "precision_seed42": E5[(mname, rep)]["precision"],
                         "false_positives_seed42": E5[(mname, rep)]["fp"]})
    return rows

t9 = pd.DataFrame(ckpt.run("tables/table9_v2", _table9))
t9["F1"] = t9.apply(lambda r: f"{r.f1_mean:.2f} +/- {r.f1_std:.2f}", axis=1)
t9["Precision"] = t9.apply(lambda r: f"{r.precision_mean:.2f} +/- {r.precision_std:.2f}", axis=1)
save_table(t9, "table09_classifier_generality",
           f"Classifier generality, mean +/- s.d. over {len(E5_SEEDS)} training draws",
           float_fmt="%.2f")
display(t9[["model", "representation", "Precision", "F1", "f1_min", "f1_max",
            "false_positives_mean", "f1_seed42"]].round(2))

save_supp(pd.DataFrame([{"seed": s, "model": m, "representation": REP_PRETTY[r],
                         **{k: E5MS[(m, r, s)][k] for k in ("precision", "recall", "f1", "fp", "tp", "fn")}}
                        for s in E5_SEEDS for m in E5_MODELS for r in ("raw10", "all18", "eng8")]),
          "table09b_classifier_generality_per_seed", "Classifier generality, every seed",
          float_fmt="%.2f")

def _e5_eng_vs_all():
    rows = []
    for m in E5_MODELS:
        a = np.array([E5MS[(m, "all18", s)]["f1"] for s in E5_SEEDS])
        e = np.array([E5MS[(m, "eng8", s)]["f1"] for s in E5_SEEDS])
        r = {"model": m, "all18_f1_mean": float(a.mean()), "eng8_f1_mean": float(e.mean()),
             "mean_gain_eng8_minus_all18": float((e - a).mean()),
             "draws_where_eng8_wins": int((e > a).sum()), "n": len(E5_SEEDS)}
        try:
            st, p = stats.wilcoxon(e, a, alternative="greater", zero_method="wilcox")
            r["wilcoxon_W"] = float(st); r["wilcoxon_p_one_sided"] = float(p)
        except Exception as ex:
            r["wilcoxon_W"] = None; r["wilcoxon_p_one_sided"] = None; r["note"] = str(ex)
        rows.append(r)
    return rows

E5CMP = pd.DataFrame(ckpt.run("E5/eng8_vs_all18_v2", _e5_eng_vs_all))
save_supp(E5CMP, "table09c_eng8_vs_all18", "Engineered 8 against CAN-SHIELD 18, paired over draws",
          float_fmt="%.4g")
display(E5CMP.round(3))
_wins = int((E5CMP["mean_gain_eng8_minus_all18"] > 0).sum())
log(f"Engineered 8 beats CAN-SHIELD 18 on mean F1 for {_wins} of {len(E5_MODELS)} classifiers")
_raw = t9[t9["representation"] == REP_PRETTY["raw10"]]
_raw_nonlr = _raw[_raw["model"] != "LogisticRegression"]
log(f"Raw 10 precision range across the non-linear classifiers (means): "
    f"{_raw_nonlr['precision_mean'].min():.1f} - {_raw_nonlr['precision_mean'].max():.1f} %")
_cs = t9[t9["representation"] == REP_PRETTY["all18"]]
log("CAN-SHIELD 18 F1 spread across draws (max - min): " +
    ", ".join(f"{r.model} {r.f1_max - r.f1_min:.1f}" for r in _cs.itertuples()))

In [ ]:
def _row_keys(X):
    A = np.ascontiguousarray(np.asarray(X, dtype=np.float32))
    return A.view(np.dtype((np.void, A.dtype.itemsize * A.shape[1]))).ravel()

def _overlap(Xtr, Xte):
    return float(np.isin(_row_keys(Xte), _row_keys(Xtr)).mean())

if RUN_DEDUP_SENSITIVITY and DUP["shared_suffix_rows"] > 0:
    log(f"Dedup sensitivity: the shared {DUP['shared_suffix_rows']:,}-frame suffix is kept once "
        f"(in {DEDUP_KEEPER}) and excluded from " +
        ", ".join(k for k in CH_ORDER if DEDUP_TAIL[k]))

    def _e1_pool_dedup():
        Xs, ys = [], []
        for ci, k in enumerate(CH_ORDER):
            Xi, yi = stratified_take(k, E1_ALLOC[k], MASTER_SEED * 1000 + 7 * ci + 13,
                                     exclude_tail=DEDUP_TAIL[k])
            Xs.append(Xi); ys.append(yi)
        return np.concatenate(Xs), np.concatenate(ys)

    X_dd, y_dd = _e1_pool_dedup()
    tr_dd, te_dd = train_test_split(np.arange(len(y_dd)), test_size=0.30,
                                    stratify=y_dd, random_state=MASTER_SEED)

    LEAK = ckpt.run("DEDUP/leakage", lambda: {
        "original_random_split_test_rows_also_in_train_pct":
            100 * _overlap(X_e1[E1_SPLITS["random"][0]], X_e1[E1_SPLITS["random"][1]]),
        "dedup_random_split_test_rows_also_in_train_pct":
            100 * _overlap(X_dd[tr_dd], X_dd[te_dd]),
        "original_chronological_test_rows_also_in_train_pct":
            100 * _overlap(X_e1[E1_SPLITS["chronological"][0]], X_e1[E1_SPLITS["chronological"][1]]),
    })
    for k, v in LEAK.items():
        log(f"   {k:55s} {v:6.2f} %")

    DD_ROWS = []
    for rep in ["raw10", "all18"]:
        for mname in E1_MODELS:
            r = ckpt.run(f"DEDUP/E1/random/{rep}/{mname}",
                         lambda mname=mname, rep=rep: fit_eval(
                             mname, X_dd[tr_dd], y_dd[tr_dd], X_dd[te_dd], y_dd[te_dd], rep,
                             n_trees=RF_E1_TREES))
            o = next((x for x in E1_RESULTS if x["protocol"] == "random"
                      and x["representation"] == rep and x["model"] == mname), None)
            DD_ROWS.append({"experiment": "E1 within-vehicle, random split", "condition": mname,
                            "representation": REP_PRETTY[rep],
                            "original_f1": o["f1"] if o else np.nan, "dedup_f1": r["f1"]})
    for comp, spec in E2_COMPOSITIONS.items():
        for rep in ["raw10", "eng8"]:
            r = ckpt.run(f"DEDUP/E2/{comp}/{rep}",
                         lambda spec=spec, rep=rep, comp=comp: run_transfer(
                             spec, rep, MASTER_SEED, tag=comp + "_dedup", dedup=True))
            DD_ROWS.append({"experiment": "E2 cross-vehicle", "condition": E2_LABEL[comp],
                            "representation": REP_PRETTY[rep],
                            "original_f1": E2[(comp, rep)]["f1"], "dedup_f1": r["f1"]})
    dd_df = pd.DataFrame(DD_ROWS)
    dd_df["difference"] = dd_df["dedup_f1"] - dd_df["original_f1"]
    save_supp(dd_df, "tableA5_dedup_sensitivity",
              "E1 and E2 re-run with the shared duplicate suffix removed", float_fmt="%.2f")
    display(dd_df.round(2))
    log(f"Largest change from removing duplicates: {dd_df['difference'].abs().max():.2f} F1 points")
else:
    log("Dedup sensitivity skipped (RUN_DEDUP_SENSITIVITY is False or no shared suffix was found).")

## 4.7 Explainability

In [ ]:
def _fit_explain_model():
    tr, te = E1_SPLITS["random"]
    Xp, yp, scaler, cw, cols = prepare(X_e1[tr], y_e1[tr], "all18", seed=MASTER_SEED)
    mdl = RandomForestClassifier(n_estimators=RF_E1_TREES, random_state=MASTER_SEED, n_jobs=-1)
    mdl.fit(Xp, yp)
    Xreal = apply_prepare(X_e1[tr], cols, scaler)
    yreal = y_e1[tr]
    return mdl, scaler, cols, Xp, yp, Xreal, yreal, te

(EXPL_MODEL, EXPL_SCALER, EXPL_COLS, EXPL_XTR, EXPL_YTR,
 EXPL_XREAL, EXPL_YREAL, EXPL_TE) = _fit_explain_model()
EXPL_XTE = apply_prepare(X_e1[EXPL_TE], EXPL_COLS, EXPL_SCALER)
EXPL_YTE = y_e1[EXPL_TE]
EXPL_CAP = cap_e1[EXPL_TE]
log(f"Explained model: RandomForest({RF_E1_TREES}) on CAN-SHIELD 18, "
    f"trained on {len(EXPL_YTR):,} balanced frames")

def _background(rs):
    n = SHAP_N_BACKGROUND
    if SHAP_BACKGROUND == "smote":
        idx = rs.choice(len(EXPL_XTR), size=min(n, len(EXPL_XTR)), replace=False)
        return EXPL_XTR[idx], EXPL_YTR[idx], "SMOTE-balanced training partition"
    if SHAP_BACKGROUND == "real_natural":
        idx = rs.choice(len(EXPL_XREAL), size=min(n, len(EXPL_XREAL)), replace=False)
        return EXPL_XREAL[idx], EXPL_YREAL[idx], "real training frames, natural prevalence"
    if SHAP_BACKGROUND == "real_balanced":
        a = np.flatnonzero(EXPL_YREAL == 1); b = np.flatnonzero(EXPL_YREAL == 0)
        idx = np.concatenate([rs.choice(a, n // 2, replace=False),
                              rs.choice(b, n - n // 2, replace=False)])
        return EXPL_XREAL[idx], EXPL_YREAL[idx], "real training frames, class balanced"
    raise KeyError(SHAP_BACKGROUND)

def mc_shapley(model, X_explain, X_background, n_perm, seed, paired=True):
    rs = np.random.RandomState(seed)
    n, d = X_explain.shape
    nb = len(X_background)
    if paired and n_perm != nb:
        raise ValueError(f"paired sampling needs n_perm == background size ({n_perm} vs {nb})")
    phi = np.zeros((n, d), dtype=np.float64)
    base_used = np.zeros(n, dtype=np.float64)
    evals = 0
    batch_rows, batch_meta = [], []

    def flush():
        nonlocal evals, batch_rows, batch_meta
        if not batch_rows:
            return
        M = np.concatenate(batch_rows, axis=0)
        p = model.predict_proba(M)[:, 1]
        evals += len(M)
        off = 0
        for (i, perm, nrows) in batch_meta:
            seq = p[off:off + nrows]
            off += nrows
            phi[i, perm] += np.diff(seq)
            base_used[i] += seq[0]
        batch_rows, batch_meta = [], []

    for i in range(n):
        x = X_explain[i]
        order = rs.permutation(nb) if paired else rs.randint(nb, size=n_perm)
        for k in range(n_perm):
            z = X_background[order[k]]
            perm = rs.permutation(d)
            rows = np.repeat(z[None, :], d + 1, axis=0)
            for kk, j in enumerate(perm):
                rows[kk + 1:, j] = x[j]
            batch_rows.append(rows)
            batch_meta.append((i, perm, d + 1))
        if sum(r.shape[0] for r in batch_rows) >= 200_000:
            flush()
    flush()
    phi /= n_perm
    base_used /= n_perm
    return phi, base_used, evals

def _shapley():
    rs = np.random.RandomState(MASTER_SEED)
    Xbg, ybg, bg_desc = _background(rs)
    ya = EXPL_YTE.astype(bool)
    n_a = SHAP_N_FRAMES // 2
    ia = rs.choice(np.flatnonzero(ya), size=min(n_a, int(ya.sum())), replace=False)
    inn = rs.choice(np.flatnonzero(~ya), size=SHAP_N_FRAMES - len(ia), replace=False)
    sel = np.sort(np.concatenate([ia, inn]))
    Xex = EXPL_XTE[sel]
    yex = EXPL_YTE[sel]

    t0 = time.time()
    phi, base_used, evals = mc_shapley(EXPL_MODEL, Xex, Xbg, SHAP_N_PERM, MASTER_SEED,
                                       paired=SHAP_PAIRED)
    secs = time.time() - t0

    fx = EXPL_MODEL.predict_proba(Xex)[:, 1]
    bg_mean = float(EXPL_MODEL.predict_proba(Xbg)[:, 1].mean())
    additivity_resid = fx - base_used - phi.sum(axis=1)
    additivity_vs_bgmean = fx - bg_mean - phi.sum(axis=1)

    return {
        "method": "paired" if SHAP_PAIRED else "random-background",
        "background_source": bg_desc,
        "background_attack_pct": float(100 * ybg.mean()),
        "background_X": Xbg.tolist(),
        "explained_index": sel.astype(int).tolist(),
        "capture": [CH_ORDER[int(c)] for c in EXPL_CAP[sel]],
        "phi": phi.tolist(),
        "feature_values": Xex.tolist(),
        "y_true": yex.astype(int).tolist(),
        "f_x": fx.tolist(),
        "baseline_per_instance": base_used.tolist(),
        "E_f_X_over_background": bg_mean,
        "n_frames": int(len(sel)), "n_perm": SHAP_N_PERM,
        "n_background": int(len(Xbg)),
        "model_evaluations": int(evals),
        "seconds": round(secs, 1),
        "additivity_max_abs_residual": float(np.abs(additivity_resid).max()),
        "additivity_vs_background_max_abs": float(np.abs(additivity_vs_bgmean).max()),
        "additivity_vs_background_mean_abs": float(np.abs(additivity_vs_bgmean).mean()),
    }

SH = ckpt.run("explain/shapley_v2", _shapley)
PHI = np.array(SH["phi"])
XEX = np.array(SH["feature_values"])
YEX = np.array(SH["y_true"])
FX = np.array(SH["f_x"])
CAPEX = np.array(SH["capture"])
XBG = np.array(SH["background_X"])
log(f"Shapley ({SH['method']}): {SH['n_frames']} frames x {SH['n_perm']} permutations "
    f"x {len(FEATURE_NAMES)} features")
log(f"   background                 : {SH['n_background']} {SH['background_source']} "
    f"({SH['background_attack_pct']:.1f}% attack)")
log(f"   model evaluations          : {SH['model_evaluations']:,}  ({SH['seconds']}s)")
log(f"   E[f(X)] over background    : {SH['E_f_X_over_background']:.4f}")
log(f"   additivity vs E[f(X)], max : {SH['additivity_vs_background_max_abs']:.2e}"
    + ("  (exact up to float error)" if SH['method'] == "paired" else "  (Monte Carlo baseline error)"))

def _treeshap():
    try:
        import shap
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "-q", "install", "shap"], check=False)
        import shap
    t0 = time.time()
    expl = shap.TreeExplainer(EXPL_MODEL, data=XBG, feature_perturbation="interventional",
                              model_output="probability")
    sv = expl.shap_values(XEX, check_additivity=False)
    if isinstance(sv, list):
        sv = sv[1]
    sv = np.asarray(sv)
    if sv.ndim == 3:
        sv = sv[:, :, 1]
    ev = expl.expected_value
    ev = float(np.ravel(ev)[-1]) if np.ndim(ev) else float(ev)
    mc, ts = np.abs(PHI).mean(axis=0), np.abs(sv).mean(axis=0)
    rho, _ = stats.spearmanr(mc, ts)
    return {"shap_version": shap.__version__, "seconds": round(time.time() - t0, 1),
            "expected_value": ev,
            "mean_abs": ts.tolist(),
            "rank_spearman_vs_mc": float(rho),
            "top_feature": FEATURE_NAMES[int(np.argmax(ts))],
            "max_abs_phi_diff": float(np.abs(sv - PHI).max()),
            "mean_abs_phi_diff": float(np.abs(sv - PHI).mean()),
            "additivity_max_abs": float(np.abs(FX - ev - sv.sum(axis=1)).max())}

TS = None
if SHAP_TREESHAP_CHECK:
    try:
        TS = ckpt.run("explain/treeshap_v2", _treeshap)
        log(f"TreeSHAP check (shap {TS['shap_version']}, {TS['seconds']}s): top feature {TS['top_feature']}, "
            f"rank agreement with Monte Carlo rho = {TS['rank_spearman_vs_mc']:.3f}, "
            f"mean |phi difference| = {TS['mean_abs_phi_diff']:.4f}")
    except Exception as e:
        log(f"TreeSHAP check skipped: {type(e).__name__}: {e}")

In [ ]:
def _perm_importance():
    rs = np.random.RandomState(MASTER_SEED)
    k = min(PERM_IMP_SAMPLE, len(EXPL_YTE))
    sel = rs.choice(len(EXPL_YTE), size=k, replace=False)
    r = permutation_importance(EXPL_MODEL, EXPL_XTE[sel], EXPL_YTE[sel],
                               n_repeats=PERM_IMP_REPEATS, random_state=MASTER_SEED,
                               scoring="f1", n_jobs=1)
    return {"mean": r.importances_mean.tolist(), "std": r.importances_std.tolist(),
            "n_sample": int(k), "n_repeats": PERM_IMP_REPEATS, "scoring": "f1"}

PI = ckpt.run("explain/permutation_importance", _perm_importance)
IMPURITY = ckpt.run("explain/impurity_importance",
                    lambda: {"mean": EXPL_MODEL.feature_importances_.tolist()})

def _table10():
    mabs = np.abs(PHI).mean(axis=0)
    pim = np.array(PI["mean"]); pis = np.array(PI["std"])
    imp = np.array(IMPURITY["mean"])
    df = pd.DataFrame({
        "feature": FEATURE_NAMES,
        "origin": [f[1] for f in FEATURES],
        "mean_abs_shapley": mabs,
        "permutation_importance": pim,
        "permutation_importance_std": pis,
        "impurity_importance": imp,
    })
    if TS is not None:
        df["mean_abs_treeshap"] = np.array(TS["mean_abs"])
        df["treeshap_rank"] = df["mean_abs_treeshap"].rank(ascending=False, method="min").astype(int)
    df["shapley_rank"] = df["mean_abs_shapley"].rank(ascending=False, method="min").astype(int)
    df["permutation_rank"] = df["permutation_importance"].rank(ascending=False, method="min").astype(int)
    df["impurity_rank"] = df["impurity_importance"].rank(ascending=False, method="min").astype(int)
    return df.sort_values("shapley_rank").to_dict("records")

t10 = pd.DataFrame(ckpt.run("tables/table10_v2", _table10))
save_table(t10, "table10_global_importance",
           "Global importance by Monte Carlo Shapley, permutation and impurity", float_fmt="%.4f")
display(t10.round(4))

top_sh = t10.iloc[0]
second = t10.iloc[1]
log(f"Top feature by Shapley     : {top_sh['feature']} ({top_sh['mean_abs_shapley']:.4f}), "
    f"{100*(top_sh['mean_abs_shapley']/second['mean_abs_shapley']-1):.0f}% ahead of "
    f"{second['feature']} ({second['mean_abs_shapley']:.4f})")
log(f"Top feature by permutation : "
    f"{t10.sort_values('permutation_rank').iloc[0]['feature']}")
log(f"Top feature by impurity    : "
    f"{t10.sort_values('impurity_rank').iloc[0]['feature']}")
if "treeshap_rank" in t10:
    log(f"Top feature by TreeSHAP    : {t10.sort_values('treeshap_rank').iloc[0]['feature']}")
_eng_in_top7 = int((t10.head(7)["origin"] == "Engineered").sum())
log(f"Engineered features in the Shapley top 7: {_eng_in_top7} of 7 "
    f"(they are 8 of {len(FEATURES)} features)")
for _f in ("CAN_ID", "ID_Freq", "IAT_ID"):
    _r = t10.set_index("feature").loc[_f]
    log(f"   {_f:8s} Shapley rank {int(_r['shapley_rank'])}, permutation rank "
        f"{int(_r['permutation_rank'])}, impurity rank {int(_r['impurity_rank'])}")

def fig10():
    d = t10.sort_values("mean_abs_shapley")
    cols = [C_ENG if o == "Engineered" else C_RAW for o in d["origin"]]
    fig, ax = plt.subplots(figsize=(8.6, 6.2))
    ax.barh(np.arange(len(d)), d["mean_abs_shapley"], color=cols, edgecolor="#333333",
            linewidth=0.6)
    ax.set_yticks(np.arange(len(d))); ax.set_yticklabels(d["feature"], fontsize=8.6)
    ax.set_xlabel("mean |Shapley contribution| to P(attack)")
    ax.set_title(f"Global feature importance by Monte Carlo Shapley sampling\n"
                 f"{SH['n_frames']} held-out frames, {SH['n_perm']} permutations "
                 f"{'paired with ' + str(SH['n_background']) + ' background frames' if SH['method']=='paired' else ''}, "
                 f"{SH['model_evaluations']:,} model evaluations", fontsize=10.2)
    from matplotlib.patches import Patch
    ax.legend(handles=[Patch(facecolor=C_ENG, label="Engineered"),
                       Patch(facecolor=C_RAW, label="Raw")], fontsize=8.6, loc="lower right")
    for i, v in enumerate(d["mean_abs_shapley"]):
        ax.text(v + max(d["mean_abs_shapley"]) * 0.012, i, f"{v:.4f}", va="center", fontsize=7.2)
    fig.tight_layout()
    return fig

save_fig(fig10(), "fig10_global_shapley")

_INT_FEATURES = {"CAN_ID", "DLC", "ID_Freq", "Payload_Hamming", "NonZero_Bytes"} | \
                {f"DATA{i}" for i in range(8)}

def _orig_units(row):
    v = np.array(row, dtype=np.float64)
    if EXPL_SCALER is not None:
        v = v * EXPL_SCALER.scale_ + EXPL_SCALER.mean_
        tol = 1e-5 * (np.abs(EXPL_SCALER.mean_) + EXPL_SCALER.scale_)
        v[np.abs(v) < tol] = 0.0
    v = np.array([float(f"{x:.6g}") for x in v])
    v[v == 0] = 0.0
    for j, f in enumerate(FEATURE_NAMES):
        if f in _INT_FEATURES:
            v[j] = float(np.round(v[j]))
    return v

def _fmt(fname, v):
    if fname in _INT_FEATURES:
        return f"{int(v)}" if fname != "CAN_ID" else f"{int(v)} (0x{int(v):03X})"
    return f"{v:.4g}"

def fig11():
    top4 = t10.head(4)["feature"].tolist()
    fig, axes = plt.subplots(2, 2, figsize=(11.4, 7.2))
    XO = np.array([_orig_units(r) for r in XEX])
    for ax, fname in zip(axes.ravel(), top4):
        j = FEATURE_NAMES.index(fname)
        xv = XO[:, j]
        for cls, col, lab in ((0, C_NORM, "normal"), (1, C_ATK, "injected")):
            m = YEX == cls
            ax.scatter(xv[m], PHI[m, j], s=15, alpha=0.65, color=col, label=lab,
                       edgecolors="none")
        ax.axhline(0, color="#777777", lw=0.8, ls="--")
        ax.set_xlabel(f"{fname} (original units)")
        ax.set_ylabel("Shapley contribution")
        ax.set_title(fname, fontsize=10)
        ax.legend(fontsize=8)
    fig.suptitle("Shapley dependence: how each leading feature's contribution varies with its value",
                 y=1.02, fontsize=11)
    fig.tight_layout()
    return fig

save_fig(fig11(), "fig11_shapley_dependence")

LOCAL_GROUPS = [("DoS", "Injected, flooding (DoS)"), ("Fuzzy", "Injected, fuzzing"),
                ("gear", "Injected, gear spoofing"), ("RPM", "Injected, RPM spoofing"),
                ("normal", "Normal traffic")]

def _local_pick():
    pred = (FX >= 0.5).astype(int)
    correct = pred == YEX
    out = {"rule": LOCAL_PICK_RULE}
    for g, _ in LOCAL_GROUPS:
        mask = (YEX == 0) if g == "normal" else ((YEX == 1) & (CAPEX == g))
        idx = np.flatnonzero(mask & correct)
        out[f"n_{g}"] = int(mask.sum()); out[f"n_correct_{g}"] = int(len(idx))
        if len(idx) == 0:
            out[g] = None
            continue
        if LOCAL_PICK_RULE == "medoid":
            P = PHI[idx]
            pick = idx[int(np.argmin(((P - P.mean(axis=0)) ** 2).sum(axis=1)))]
        else:
            pick = idx[int(np.argmax(FX[idx]))] if g != "normal" else idx[int(np.argmin(FX[idx]))]
        out[g] = int(pick)
    return out

LOCAL = ckpt.run("explain/local_pick_v2", _local_pick)
log(f"Local examples ({LOCAL['rule']} rule): " +
    ", ".join(f"{g}: {LOCAL['n_correct_' + g]}/{LOCAL['n_' + g]} correct" for g, _ in LOCAL_GROUPS))

def _table11():
    rows = []
    for g, label in LOCAL_GROUPS:
        idx = LOCAL.get(g)
        if idx is None:
            continue
        vals = _orig_units(XEX[idx])
        order = np.argsort(-np.abs(PHI[idx]))
        for rank, j in enumerate(order, 1):
            rows.append({"frame": label, "capture": str(CAPEX[idx]), "p_attack": float(FX[idx]),
                         "baseline_E_f_X": SH["E_f_X_over_background"],
                         "sum_of_contributions": float(PHI[idx].sum()),
                         "rank": rank, "feature": FEATURE_NAMES[j], "value": float(vals[j]),
                         "contribution": float(PHI[idx, j])})
    return rows

t11 = pd.DataFrame(ckpt.run("tables/table11_v2", _table11))
save_table(t11, "table11_local_shapley",
           f"Per-feature Shapley contributions for one representative frame per group "
           f"({LOCAL_PICK_RULE} rule)")
display(t11.groupby("frame", sort=False).head(6).round(4))

def fig12(top=8):
    fig, axes = plt.subplots(2, 3, figsize=(16.5, 8.6))
    axl = axes.ravel()
    for ax, (g, label) in zip(axl, LOCAL_GROUPS):
        idx = LOCAL.get(g)
        if idx is None:
            ax.axis("off"); continue
        vals = _orig_units(XEX[idx])
        order = np.argsort(np.abs(PHI[idx]))[-top:]
        c = [C_ATK if PHI[idx, j] > 0 else C_ENG for j in order]
        ax.barh(np.arange(len(order)), PHI[idx, order], color=c, edgecolor="#333333",
                linewidth=0.6)
        ax.set_yticks(np.arange(len(order)))
        ax.set_yticklabels([f"{FEATURE_NAMES[j]} = {_fmt(FEATURE_NAMES[j], vals[j])}"
                            for j in order], fontsize=8.2)
        ax.axvline(0, color="#444444", lw=0.9)
        ax.set_xlabel("Shapley contribution to P(attack)", fontsize=8.6)
        ax.set_title(f"{label}\nP(attack) = {FX[idx]:.3f}   E[f(X)] = {SH['E_f_X_over_background']:.3f}   "
                     f"sum = {PHI[idx].sum():+.3f}", fontsize=9.2)
    for ax in axl[len(LOCAL_GROUPS):]:
        ax.axis("off")
        ax.text(0.02, 0.5,
                f"One correctly classified frame per group,\nchosen as the {LOCAL_PICK_RULE} of the "
                f"group's Shapley vectors\n(the most typical explanation).\n\n"
                f"Red pushes toward attack, teal toward normal.\nContributions sum to "
                f"P(attack) - E[f(X)].", fontsize=9, va="center", transform=ax.transAxes)
    fig.suptitle("Local Shapley explanations, one representative frame per attack class and for normal traffic",
                 y=1.01, fontsize=11)
    fig.tight_layout()
    return fig

save_fig(fig12(), "fig12_local_shapley")

## 6.1 OTIDS labelling audit

In [ ]:
IMPERSONATION_ID = 0x164
IMPERSONATION_BOUNDARY = 250.0

def _otids_fuzzy_audit():
    af = set(VOCAB["ot_attack_free"]["normal_ids"])
    fz = set(VOCAB["ot_fuzzy"]["normal_ids"]) | set(VOCAB["ot_fuzzy"]["attack_ids"])
    novel = sorted(fz - af)
    X, y = OT["fuzzy"]
    cid = np.asarray(X[:, 0]).astype(np.int32)
    labelled = int(np.isin(cid, list(novel)).sum()) if novel else 0
    return {"attack_free_distinct_ids": len(af),
            "fuzzy_distinct_ids": len(fz),
            "fuzzy_ids_absent_from_attack_free": len(novel),
            "novel_ids": [f"0x{i:03X}" for i in novel],
            "frames_the_documented_rule_would_label": labelled,
            "fuzzy_capture_frames": int(len(cid)),
            "rule_is_applicable": labelled > 0}

FZA = ckpt.run("section61/fuzzy_rule", _otids_fuzzy_audit)
log("OTIDS fuzzy labelling rule:")
log(f"   attack-free capture distinct identifiers : {FZA['attack_free_distinct_ids']}")
log(f"   fuzzy capture distinct identifiers       : {FZA['fuzzy_distinct_ids']}")
log(f"   fuzzy identifiers absent from attack-free: {FZA['fuzzy_ids_absent_from_attack_free']}")
log(f"   frames the documented rule would label   : "
    f"{FZA['frames_the_documented_rule_would_label']:,} of {FZA['fuzzy_capture_frames']:,}")
log(f"   => rule applicable to the released files : {FZA['rule_is_applicable']}")

def _otids_impersonation_audit():
    if not HAVE_IMPERSONATION:
        return {"available": False,
                "note": ("Impersonation_attack_dataset.txt is not present in data/. "
                         "This claim cannot be verified from the supplied files; place the "
                         "capture in data/ and re-run with FORCE_RECOMPUTE=['section61'].")}
    X, y = OT["impersonation"]
    cid = np.asarray(X[:, 0]).astype(np.int32)
    ts = np.asarray(OT_TS["impersonation"], dtype=np.float64)
    rel = ts - ts.min()
    m = cid == IMPERSONATION_ID
    before = m & (rel < IMPERSONATION_BOUNDARY)
    after = m & (rel >= IMPERSONATION_BOUNDARY)
    dur_b = IMPERSONATION_BOUNDARY
    dur_a = max(rel.max() - IMPERSONATION_BOUNDARY, 1e-9)
    rate_b = before.sum() / dur_b
    rate_a = after.sum() / dur_a
    sec = np.floor(rel[m]).astype(np.int64)
    counts = np.bincount(sec, minlength=int(np.ceil(rel.max())) + 1)
    rb = counts[:int(IMPERSONATION_BOUNDARY)]
    ra = counts[int(IMPERSONATION_BOUNDARY):]
    out = {"available": True,
           "capture_frames": int(len(ts)),
           "id": f"0x{IMPERSONATION_ID:03X}",
           "frames_before": int(before.sum()), "frames_after": int(after.sum()),
           "rate_before_fps": float(rate_b), "rate_after_fps": float(rate_a),
           "rate_difference_fps": float(rate_b - rate_a),
           "per_second_mean_before": float(rb.mean()) if len(rb) else None,
           "per_second_mean_after": float(ra.mean()) if len(ra) else None,
           "recording_duration_s": float(rel.max())}
    if len(rb) > 1 and len(ra) > 1:
        st, p = stats.mannwhitneyu(rb, ra, alternative="two-sided")
        out["mannwhitney_U"] = float(st)
        out["mannwhitney_p"] = float(p)
        out["populations_distinguishable_at_0.05"] = bool(p < 0.05)
    return out

IMP = ckpt.run("section61/impersonation_rule", _otids_impersonation_audit)
log("OTIDS impersonation labelling rule:")
if not IMP.get("available"):
    log("   " + IMP["note"])
else:
    log(f"   {IMP['id']} frame rate before {IMPERSONATION_BOUNDARY:.0f}s : "
        f"{IMP['rate_before_fps']:.1f} fps ({IMP['frames_before']:,} frames)")
    log(f"   {IMP['id']} frame rate after  {IMPERSONATION_BOUNDARY:.0f}s : "
        f"{IMP['rate_after_fps']:.1f} fps ({IMP['frames_after']:,} frames)")
    log(f"   difference                            : {IMP['rate_difference_fps']:+.2f} fps")
    if "mannwhitney_p" in IMP:
        log(f"   Mann-Whitney U on per-second counts    : "
            f"U = {IMP['mannwhitney_U']:.1f}, p = {IMP['mannwhitney_p']:.4f} "
            f"-> distinguishable at 0.05: {IMP['populations_distinguishable_at_0.05']}")

rows = [{"capture": "fuzzy", "documented_rule": "identifier absent from the attack-free capture",
         "frames_the_rule_labels": FZA["frames_the_documented_rule_would_label"],
         "rule_reproducible": FZA["rule_is_applicable"], "detail":
         f"{FZA['fuzzy_distinct_ids']} vs {FZA['attack_free_distinct_ids']} distinct identifiers, "
         f"{FZA['fuzzy_ids_absent_from_attack_free']} novel"}]
if IMP.get("available"):
    rows.append({"capture": "impersonation",
                 "documented_rule": f"{IMP['id']} injected after {IMPERSONATION_BOUNDARY:.0f}s",
                 "frames_the_rule_labels": IMP["frames_after"],
                 "rule_reproducible": IMP.get("populations_distinguishable_at_0.05"),
                 "detail": f"{IMP['rate_before_fps']:.1f} fps before vs "
                           f"{IMP['rate_after_fps']:.1f} fps after"})
else:
    rows.append({"capture": "impersonation", "documented_rule":
                 f"{IMPERSONATION_ID:#05x} injected after {IMPERSONATION_BOUNDARY:.0f}s",
                 "frames_the_rule_labels": None, "rule_reproducible": None,
                 "detail": "capture file not supplied - not verifiable in this run"})
save_supp(pd.DataFrame(rows), "table12_otids_label_audit",
           "Reproducibility of the published OTIDS labelling rules", float_fmt="%.2f")
display(pd.DataFrame(rows))

## Comparison with the paper

In [ ]:
PAPER_REPORTED = {
    "ch_DoS_frames": 3665771, "ch_DoS_attack": 587521, "ch_DoS_attack_pct": 16.03, "ch_DoS_attack_ids": 1,
    "ch_Fuzzy_frames": 3838860, "ch_Fuzzy_attack": 491847, "ch_Fuzzy_attack_pct": 12.81, "ch_Fuzzy_attack_ids": 2048,
    "ch_gear_frames": 4443142, "ch_gear_attack": 597252, "ch_gear_attack_pct": 13.44, "ch_gear_attack_ids": 1,
    "ch_RPM_frames": 4621702, "ch_RPM_attack": 654897, "ch_RPM_attack_pct": 14.17, "ch_RPM_attack_ids": 1,
    "ch_combined_frames": 16569475, "ch_combined_attack": 2331517, "ch_combined_attack_pct": 14.07,
    "otids_test_frames": 2573117, "otids_test_attack": 335624, "otids_test_attack_pct": 13.04,
    "ch_normal_ids": 38, "otids_normal_ids": 45, "shared_normal_ids": 22,
    "cv_LogisticRegression_raw": 72.16, "cv_LogisticRegression_cs": 97.79,
    "cv_KNeighbours_raw": 99.62, "cv_KNeighbours_cs": 99.94,
    "cv_DecisionTree_raw": 99.73, "cv_DecisionTree_cs": 99.94,
    "cv_RandomForest_raw": 100.00, "cv_RandomForest_cs": 99.98,
    "cv_AdaBoost_raw": 99.41, "cv_AdaBoost_cs": 99.96,
    "cv_HistGradientBoosting_raw": 99.98, "cv_HistGradientBoosting_cs": 99.95,
    "cv_wilcoxon_W": 18.0, "cv_wilcoxon_p": 0.078, "cv_mean_delta": 4.44,
    "e2_flood_raw_f1": 100.00, "e2_flood_raw_prec": 100.00,
    "e2_flood_eng_f1": 96.89, "e2_flood_eng_prec": 93.97,
    "e2_three_raw_f1": 100.00, "e2_three_eng_f1": 96.70,
    "e2_all4_raw_f1": 42.28, "e2_all4_raw_prec": 26.81,
    "e2_all4_eng_f1": 97.87, "e2_all4_eng_prec": 95.82,
    "e2_all4_train_attack_ids": 1743,
    "e4_raw_f1": 87.52, "e4_raw_prec": 77.81, "e4_eng_f1": 99.69, "e4_eng_prec": 100.00,
    "draws_raw_nofuzz_f1": 100.00, "draws_raw_fuzz_f1": 42.48, "draws_raw_fuzz_f1_sd": 1.40,
    "draws_raw_fuzz_prec": 26.98, "draws_eng_nofuzz_f1": 97.15, "draws_eng_fuzz_f1": 97.53,
    "draws_wilcoxon_W": 15.0, "draws_wilcoxon_p": 0.031,
    "dose_0_raw_f1": 100.00, "dose_800_raw_f1": 77.97, "dose_2000_raw_f1": 62.64,
    "dose_4000_raw_f1": 62.06, "dose_10000_raw_f1": 56.16, "dose_20000_raw_f1": 49.43,
    "dose_40000_raw_f1": 42.86,
    "dose_0_ids": 3, "dose_800_ids": 105, "dose_2000_ids": 241, "dose_4000_ids": 458,
    "dose_10000_ids": 960, "dose_20000_ids": 1470, "dose_40000_ids": 1886,
    "dose_eng_min": 96.96, "dose_eng_max": 98.09,
    "e5_LogisticRegression_raw10": 0.00, "e5_LogisticRegression_all18": 82.26, "e5_LogisticRegression_eng8": 89.72,
    "e5_DecisionTree_raw10": 46.61, "e5_DecisionTree_all18": 93.75, "e5_DecisionTree_eng8": 95.90,
    "e5_RandomForest_raw10": 45.30, "e5_RandomForest_all18": 91.17, "e5_RandomForest_eng8": 97.79,
    "e5_HistGradientBoosting_raw10": 45.94, "e5_HistGradientBoosting_all18": 99.58, "e5_HistGradientBoosting_eng8": 98.33,
    "t5_ID_Freq_normal": 3.6190, "t5_ID_Freq_attack": 21.2780,
    "t5_IAT_ID_normal": 0.0730, "t5_IAT_ID_attack": 2.0930,
    "t5_Payload_Entropy_normal": 1.8900, "t5_Payload_Entropy_attack": 1.8810,
    "t5_ID_Freq_max": 48.0,
    "shap_top_feature": "ID_Freq", "shap_ID_Freq": 0.0693, "shap_CAN_ID": 0.0488,
    "shap_E_f_X": 0.4849, "perm_ID_Freq": 0.0166, "impurity_top_feature": "IAT_ID",
    "chrono_test_attack_pct": 4.43,
    "chrono_DecisionTree_raw": 85.63, "prev_DecisionTree_raw": 94.94,
    "chrono_RandomForest_raw": 100.00, "prev_RandomForest_raw": 100.00,
    "bal_all4_raw_diff": 0.38,
    "dose_monotonic": "yes",
    "local_fuzzy_canid_sign": "negative",
    "otids_fuzzy_novel_ids": 0, "otids_fuzzy_frames_labelled": 0,
}

TOL = {"exact": 0.0, "tight": 2.5, "loose": 12.0, "monte_carlo": 60.0}

CHECKS = []
def chk(section, quantity, paper_key, reproduced, tol="tight", unit="", note=""):
    paper = PAPER_REPORTED.get(paper_key)
    row = {"section": section, "quantity": quantity, "paper": paper,
           "reproduced": reproduced, "unit": unit, "tolerance": tol, "note": note}
    if isinstance(paper, str) or isinstance(reproduced, str):
        row["abs_diff"] = None
        row["rel_diff_pct"] = None
        row["verdict"] = "MATCH" if paper == reproduced else "DIFFERS"
    elif paper is None or reproduced is None:
        row["abs_diff"] = None; row["rel_diff_pct"] = None; row["verdict"] = "NOT CHECKED"
    else:
        d = abs(float(reproduced) - float(paper))
        row["abs_diff"] = d
        row["rel_diff_pct"] = (100 * d / abs(float(paper))) if float(paper) else None
        if tol == "exact":
            row["verdict"] = "EXACT" if d == 0 else "DIFFERS"
        else:
            lim = TOL[tol]
            rel = row["rel_diff_pct"]
            row["verdict"] = "WITHIN TOLERANCE" if (d <= lim or (rel is not None and rel <= lim)) \
                             else "OUTSIDE TOLERANCE"
    CHECKS.append(row)

for k in CH_ORDER:
    d = VOCAB[f"ch_{k}"]
    chk("Table 2", f"{k} frames", f"ch_{k}_frames", d["frames"], "exact", "frames")
    chk("Table 2", f"{k} attack frames", f"ch_{k}_attack", d["attack_frames"], "exact", "frames")
    chk("Table 2", f"{k} attack per cent", f"ch_{k}_attack_pct", round(d["attack_pct"], 2), "tight", "%")
    chk("Table 2", f"{k} distinct attack ids", f"ch_{k}_attack_ids", len(d["attack_ids"]), "exact", "ids")
d = VOCAB["ch_combined"]
chk("Table 2", "combined frames", "ch_combined_frames", d["frames"], "exact", "frames")
chk("Table 2", "combined attack frames", "ch_combined_attack", d["attack_frames"], "exact", "frames")
chk("Table 2", "combined attack per cent", "ch_combined_attack_pct", round(d["attack_pct"], 2), "tight", "%")
chk("Table 2", "OTIDS test frames", "otids_test_frames", OTIDS_TEST["frames"], "exact", "frames")
chk("Table 2", "OTIDS test attack frames", "otids_test_attack", OTIDS_TEST["attack_frames"], "exact", "frames")
chk("Table 2", "OTIDS test attack per cent", "otids_test_attack_pct", round(OTIDS_TEST["attack_pct"], 2), "tight", "%")

_ov = {r["car_hacking_scope"]: r for r in OVERLAP}
_comb = _ov["combined (all four captures)"]
chk("Section 3.2", "Car-Hacking normal identifiers", "ch_normal_ids", _comb["ch_normal_ids"], "exact", "ids")
chk("Section 3.2", "OTIDS normal identifiers", "otids_normal_ids", _comb["otids_normal_ids"], "exact", "ids")
chk("Section 3.2", "shared normal identifiers", "shared_normal_ids", _comb["shared"], "exact", "ids")

_t5 = {r["feature"]: r for r in t5.to_dict("records")}
for f in ("ID_Freq", "IAT_ID", "Payload_Entropy"):
    chk("Table 5", f"{f} mean | normal", f"t5_{f}_normal", round(_t5[f]["mean_given_normal"], 4), "loose")
    chk("Table 5", f"{f} mean | attack", f"t5_{f}_attack", round(_t5[f]["mean_given_attack"], 4), "loose")
chk("Table 5", "ID_Freq max", "t5_ID_Freq_max", round(_t5["ID_Freq"]["max"], 2), "loose")

for m in E1_MODELS:
    chk("Table 6", f"{m} CV F1, raw", f"cv_{m}_raw", round(CV[(m, 'raw10')]["f1_mean"], 2), "tight", "%")
    chk("Table 6", f"{m} CV F1, CAN-SHIELD", f"cv_{m}_cs", round(CV[(m, 'all18')]["f1_mean"], 2), "tight", "%")
chk("Table 6", "Wilcoxon W", "cv_wilcoxon_W", W1["W"], "loose")
chk("Table 6", "Wilcoxon p (one-sided)", "cv_wilcoxon_p", W1["p_one_sided"], "loose")
chk("Table 6", "mean delta across models", "cv_mean_delta", round(W1["mean_delta"], 2), "loose", "pp")

_e1w = {(r["protocol"], r["model"]): r for r in e1_wide.to_dict("records")}
_prev = {r["protocol"]: r for r in t6b.to_dict("records")}
chk("Section 4.2", "chronological test attack per cent", "chrono_test_attack_pct",
    round(_prev["chronological"]["test_attack_pct"], 2), "tight", "%")
for _proto, _tag in (("chronological", "chrono"), ("prevalence_matched", "prev")):
    for _m in ("DecisionTree", "RandomForest"):
        if (_proto, _m) in _e1w:
            chk("Section 4.2", f"{_proto}, {_m} raw F1", f"{_tag}_{_m}_raw",
                round(_e1w[(_proto, _m)]["raw10"], 2), "tight", "%")

chk("Table 7", "flooding only, raw F1", "e2_flood_raw_f1", round(E2[("flooding_only","raw10")]["f1"], 2), "tight", "%")
chk("Table 7", "flooding only, raw precision", "e2_flood_raw_prec", round(E2[("flooding_only","raw10")]["precision"], 2), "tight", "%")
chk("Table 7", "flooding only, engineered F1", "e2_flood_eng_f1", round(E2[("flooding_only","eng8")]["f1"], 2), "tight", "%")
chk("Table 7", "flooding+gear+RPM, raw F1", "e2_three_raw_f1", round(E2[("flooding_gear_rpm","raw10")]["f1"], 2), "tight", "%")
chk("Table 7", "flooding+gear+RPM, engineered F1", "e2_three_eng_f1", round(E2[("flooding_gear_rpm","eng8")]["f1"], 2), "tight", "%")
chk("Table 7", "all four, raw F1", "e2_all4_raw_f1", round(E2[("all_four_with_fuzzing","raw10")]["f1"], 2), "tight", "%")
chk("Table 7", "all four, raw precision", "e2_all4_raw_prec", round(E2[("all_four_with_fuzzing","raw10")]["precision"], 2), "tight", "%")
chk("Table 7", "all four, engineered F1", "e2_all4_eng_f1", round(E2[("all_four_with_fuzzing","eng8")]["f1"], 2), "tight", "%")
chk("Table 7", "all four, training attack ids", "e2_all4_train_attack_ids",
    E2[("all_four_with_fuzzing","raw10")]["training_attack_ids"], "loose", "ids")
chk("Table 7", "reverse transfer, raw F1", "e4_raw_f1", round(E4["raw10"]["f1"], 2), "tight", "%")
chk("Table 7", "reverse transfer, raw precision", "e4_raw_prec", round(E4["raw10"]["precision"], 2), "tight", "%")
chk("Table 7", "reverse transfer, engineered F1", "e4_eng_f1", round(E4["eng8"]["f1"], 2), "tight", "%")

_dr = {(r["condition"], r["representation"]): r for r in t_e3b.to_dict("records")}
chk("Section 4.3", "five draws, raw no-fuzzing F1", "draws_raw_nofuzz_f1",
    round(_dr[("no_fuzzing","raw10")]["f1_mean"], 2), "tight", "%")
chk("Section 4.3", "five draws, raw with-fuzzing F1", "draws_raw_fuzz_f1",
    round(_dr[("with_fuzzing","raw10")]["f1_mean"], 2), "tight", "%")
chk("Section 4.3", "five draws, raw with-fuzzing F1 s.d.", "draws_raw_fuzz_f1_sd",
    round(_dr[("with_fuzzing","raw10")]["f1_std"], 2), "loose", "%")
chk("Section 4.3", "five draws, raw with-fuzzing precision", "draws_raw_fuzz_prec",
    round(_dr[("with_fuzzing","raw10")]["precision_mean"], 2), "tight", "%")
chk("Section 4.3", "five draws, engineered no-fuzzing F1", "draws_eng_nofuzz_f1",
    round(_dr[("no_fuzzing","eng8")]["f1_mean"], 2), "tight", "%")
chk("Section 4.3", "five draws, engineered with-fuzzing F1", "draws_eng_fuzz_f1",
    round(_dr[("with_fuzzing","eng8")]["f1_mean"], 2), "tight", "%")
chk("Section 4.3", "five draws Wilcoxon W", "draws_wilcoxon_W", W2["raw10"]["W"], "loose")
chk("Section 4.3", "five draws Wilcoxon p", "draws_wilcoxon_p", W2["raw10"]["p_one_sided"], "loose")

_tb = {(r["condition"], r["representation"]): r for r in t_bal.to_dict("records")}
_b = _tb.get((E2_LABEL["all_four_with_fuzzing"], REP_PRETTY["raw10"]))
if _b:
    chk("Section 4.3", "SMOTE vs class weighting, all four, |raw F1 difference|", "bal_all4_raw_diff",
        round(abs(_b["difference"]), 2), "tight", "pp")

_t8 = {int(r["fuzzing_frames_added"]): r for r in t8.to_dict("records")}
for nfr in (0, 800, 2000, 4000, 10000, 20000, 40000):
    if nfr in _t8:
        chk("Table 8", f"dose {nfr} frames, raw F1 (seed 42)", f"dose_{nfr}_raw_f1",
            round(_t8[nfr]["raw_f1_seed42"], 2), "tight", "%")
        chk("Table 8", f"dose {nfr} frames, raw F1 (mean of {len(E3_SEEDS)})", f"dose_{nfr}_raw_f1",
            round(_t8[nfr]["raw_f1_mean"], 2), "tight", "%", note=f"s.d. {_t8[nfr]['raw_f1_std']:.2f}")
        chk("Table 8", f"dose {nfr} frames, attack ids (seed 42)", f"dose_{nfr}_ids",
            int(_t8[nfr]["distinct_attack_ids_seed42"]), "loose", "ids")
chk("Table 8", "engineered 8 minimum across sweep (seed 42)", "dose_eng_min",
    round(t8["eng8_f1_seed42"].min(), 2), "tight", "%")
chk("Table 8", "engineered 8 maximum across sweep (seed 42)", "dose_eng_max",
    round(t8["eng8_f1_seed42"].max(), 2), "tight", "%")
chk("Table 8", "raw F1 falls with dose (Page's L trend test, p < 0.05)", "dose_monotonic",
    "yes" if TREND.get("raw_page_p", 1.0) < 0.05 else "no",
    note=f"p = {TREND.get('raw_page_p')}; mean curve strictly monotonic: "
         f"{TREND['raw_mean_curve_monotonic_non_increasing']}")

for m in E5_MODELS:
    for rep in ("raw10", "all18", "eng8"):
        chk("Table 9", f"{m} / {REP_PRETTY[rep]} F1 (seed 42)", f"e5_{m}_{rep}",
            round(E5[(m, rep)]["f1"], 2), "tight", "%")
        _r9 = t9[(t9["model"] == m) & (t9["representation"] == REP_PRETTY[rep])].iloc[0]
        chk("Table 9", f"{m} / {REP_PRETTY[rep]} F1 (mean of {len(E5_SEEDS)})", f"e5_{m}_{rep}",
            round(float(_r9["f1_mean"]), 2), "tight", "%",
            note=f"s.d. {_r9['f1_std']:.2f}, range {_r9['f1_min']:.2f}-{_r9['f1_max']:.2f}")

chk("Table 10", "top feature by Shapley", "shap_top_feature", str(t10.iloc[0]["feature"]))
chk("Table 10", "ID_Freq mean |Shapley|", "shap_ID_Freq",
    round(float(t10.set_index("feature").loc["ID_Freq", "mean_abs_shapley"]), 4), "monte_carlo")
chk("Table 10", "CAN_ID mean |Shapley|", "shap_CAN_ID",
    round(float(t10.set_index("feature").loc["CAN_ID", "mean_abs_shapley"]), 4), "monte_carlo")
chk("Section 4.7", "E[f(X)] over background", "shap_E_f_X", round(SH["E_f_X_over_background"], 4), "loose")
chk("Section 4.7", "ID_Freq permutation importance", "perm_ID_Freq",
    round(float(t10.set_index("feature").loc["ID_Freq", "permutation_importance"]), 4), "monte_carlo")
chk("Section 4.7", "top feature by impurity", "impurity_top_feature",
    str(t10.sort_values("impurity_rank").iloc[0]["feature"]))
_lf = t11[(t11["capture"] == "Fuzzy") & (t11["feature"] == "CAN_ID")]
if len(_lf):
    chk("Section 4.7", "CAN_ID contribution sign, representative fuzzing frame", "local_fuzzy_canid_sign",
        "negative" if float(_lf["contribution"].iloc[0]) < 0 else "positive",
        note=f"contribution {float(_lf['contribution'].iloc[0]):+.4f}")

chk("Section 6.1", "OTIDS fuzzy identifiers absent from attack-free", "otids_fuzzy_novel_ids",
    FZA["fuzzy_ids_absent_from_attack_free"], "exact", "ids")
chk("Section 6.1", "frames the fuzzy rule labels", "otids_fuzzy_frames_labelled",
    FZA["frames_the_documented_rule_would_label"], "exact", "frames")

val = pd.DataFrame(CHECKS)
save_supp(val, "table99_validation_report",
           "Reproduced values against the values published in the paper")

summary = val["verdict"].value_counts().rename_axis("verdict").reset_index(name="count")
save_supp(summary, "table99b_validation_summary", "Validation verdict counts", float_fmt="%.0f")
log("=" * 78)
log("VALIDATION SUMMARY")
for _, r in summary.iterrows():
    log(f"   {r['verdict']:20s} {r['count']:>4d}")
log("=" * 78)
bad = val[val["verdict"].isin(["DIFFERS", "OUTSIDE TOLERANCE"])]
if len(bad):
    log(f"{len(bad)} quantities fall outside tolerance - inspect these individually:")
    display(bad[["section", "quantity", "paper", "reproduced", "abs_diff", "rel_diff_pct", "verdict"]].round(4))
else:
    log("Every checked quantity reproduces within its stated tolerance.")
display(val.round(4))

In [ ]:
def _listing(d, exts):
    out = []
    for f in sorted(os.listdir(d)):
        if f.lower().endswith(exts):
            out.append({"file": f, "bytes": os.path.getsize(os.path.join(d, f))})
    return out

MANIFEST = {
    "environment": ENV,
    "utc_finish": datetime.now(timezone.utc).isoformat(timespec="seconds"),
    "config": {
        "MASTER_SEED": MASTER_SEED, "MULTI_SEEDS": MULTI_SEEDS, "WINDOW": WINDOW,
        "E1_POOL": E1_POOL, "E1_CV_POOL": E1_CV_POOL, "E2_TOTAL": E2_TOTAL,
        "E3_BASE_EACH": E3_BASE_EACH, "E3_FUZZ_SUB": E3_FUZZ_SUB, "E3_DOSES": E3_DOSES,
        "E5_TOTAL": E5_TOTAL, "RF_TRANSFER_TREES": RF_TRANSFER_TREES,
        "RF_E1_TREES": RF_E1_TREES, "OTIDS_TEST_TOTAL": OTIDS_TEST_TOTAL,
        "SHAP_N_FRAMES": SHAP_N_FRAMES, "SHAP_N_PERM": SHAP_N_PERM,
        "SHAP_N_BACKGROUND": SHAP_N_BACKGROUND, "RUN_SLOW_E1": RUN_SLOW_E1,
        "SHAP_PAIRED": SHAP_PAIRED, "SHAP_BACKGROUND": SHAP_BACKGROUND,
        "SHAP_TREESHAP_CHECK": SHAP_TREESHAP_CHECK, "LOCAL_PICK_RULE": LOCAL_PICK_RULE,
        "E3_SEEDS": E3_SEEDS, "E5_SEEDS": E5_SEEDS,
        "RUN_DEDUP_SENSITIVITY": RUN_DEDUP_SENSITIVITY, "DEDUP_KEEPER": DEDUP_KEEPER,
        "balance_policy": BALANCE_POLICY, "scale_policy": SCALE_POLICY,
    },
    "allocations": {"E1": E1_ALLOC, "E1_CV": E1_CV_ALLOC, "E5": E5_ALLOC},
    "datasets": {k: {"frames": VOCAB[k]["frames"], "attack_frames": VOCAB[k]["attack_frames"],
                     "attack_pct": round(VOCAB[k]["attack_pct"], 4),
                     "distinct_attack_ids": len(VOCAB[k]["attack_ids"]),
                     "distinct_normal_ids": len(VOCAB[k]["normal_ids"])}
                 for k in VOCAB},
    "otids_test_set": OTIDS_TEST,
    "integrity": {"duplication": {k: v for k, v in DUP.items() if k != "pairwise"}},
    "validation": {r["verdict"]: int(c) for r, c in
                   zip(summary.to_dict("records"), summary["count"])} if len(summary) else {},
    "checkpoint_units": len(ckpt.manifest),
    "total_checkpoint_seconds": round(sum(v.get("seconds", 0) for v in ckpt.manifest.values()), 1),
    "tables": _listing(TABLE_DIR, (".csv",)),
    "figures": _listing(FIG_DIR, (".png",)),
    "supplementary": _listing(SUPP_DIR, (".csv",)) if EMIT_SUPPLEMENTARY else [],
    "cache": _listing(CACHE_DIR, (".npy",)),
}
with open(os.path.join(RESULTS, "run_manifest.json"), "w", encoding="utf-8") as f:
    json.dump(MANIFEST, f, indent=2, cls=NpEncoder)

log("=" * 78)
log("RUN COMPLETE")
log(f"   checkpoint units        : {MANIFEST['checkpoint_units']}")
log(f"   total compute recorded  : {MANIFEST['total_checkpoint_seconds']/3600:.2f} h")
log(f"   tables written          : {len(MANIFEST['tables'])}  (expected 8: Table 2, 5-11)")
log(f"   figures written         : {len(MANIFEST['figures'])}  (expected 11: Figure 2-12)")
log(f"   supplementary files     : {len(MANIFEST['supplementary'])}")
log(f"   feature cache           : "
    f"{sum(c['bytes'] for c in MANIFEST['cache'])/1e9:.2f} GB in {len(MANIFEST['cache'])} files")
log(f"   manifest                : {os.path.join(RESULTS,'run_manifest.json')}")
log("=" * 78)

EXPECTED_TABLES = ["table02_dataset_composition", "table05_engineered_feature_stats",
                   "table06_within_vehicle_cv", "table07_cross_vehicle_transfer",
                   "table08_dose_response", "table09_classifier_generality",
                   "table10_global_importance", "table11_local_shapley"]
EXPECTED_FIGURES = ["fig02_capture_composition", "fig03_identifier_frequency",
                    "fig04_class_conditional_distributions", "fig05_feature_correlation",
                    "fig06_within_vehicle_cv_f1", "fig07_composition_ablation",
                    "fig08_five_independent_draws", "fig09_dose_response",
                    "fig10_global_shapley", "fig11_shapley_dependence",
                    "fig12_local_shapley"]
got_t = sorted(t["file"][:-4] for t in MANIFEST["tables"])
got_f = sorted(g["file"][:-4] for g in MANIFEST["figures"])
log("OUTPUT SET CHECK")
log(f"   tables  missing : {sorted(set(EXPECTED_TABLES) - set(got_t)) or 'none'}")
log(f"   tables  extra   : {sorted(set(got_t) - set(EXPECTED_TABLES)) or 'none'}")
log(f"   figures missing : {sorted(set(EXPECTED_FIGURES) - set(got_f)) or 'none'}")
log(f"   figures extra   : {sorted(set(got_f) - set(EXPECTED_FIGURES)) or 'none'}")
log("=" * 78)

idx = []
for t in MANIFEST["tables"]:
    idx.append({"kind": "table", "file": "tables/" + t["file"], "KB": round(t["bytes"]/1e3, 1)})
for g in MANIFEST["figures"]:
    idx.append({"kind": "figure", "file": "figures/" + g["file"], "KB": round(g["bytes"]/1e3, 1)})
idx_df = pd.DataFrame(idx)
display(idx_df)

In [ ]:
DELETE_CACHE = False

if DELETE_CACHE:
    n = tot = 0
    for f in os.listdir(CACHE_DIR):
        p = os.path.join(CACHE_DIR, f)
        tot += os.path.getsize(p)
        os.remove(p)
        n += 1
    log(f"Removed {n} cache files, {tot/1e9:.2f} GB freed.")
else:
    tot = sum(os.path.getsize(os.path.join(CACHE_DIR, f)) for f in os.listdir(CACHE_DIR))
    log(f"Feature cache retained: {tot/1e9:.2f} GB in {CACHE_DIR}")
    log("Set DELETE_CACHE = True in this cell to reclaim the space once you are finished.")